# Image Generation

> 🎯 **本章學完你將能學會什麼：**
> - 理解 OpenAI 的 **GPT-Image-2** 模型特色與使用方式  
> - 熟悉 `client.images.generate()` 與 `client.images.edit()` 的使用方法  
> - 掌握如何設定影像生成的 **size**、**quality**、**moderation** 等參數  
> - 能以英文 prompt 生成高品質 AI 圖像  
> - 瞭解如何將 base64 圖像資料轉換、顯示與儲存  

OpenAI 提供 文生圖 (text to image) 和 圖生圖 (image to image) API。

## GPT-Image-2

OpenAI 最新的文生圖模型 (2026-04-21 發布)

優點:
- 品質更好 (quality `medium` 的結果 比 Image-1.5 quality `high` 更好)
- 價格更便宜 
- 可以穩定出文字(英文)

建議使用英文作為Prompt。

## 📚 延伸資源與參考連結

> 這些連結可幫助你更深入理解 OpenAI API 與 LangChain 的應用方式，  
> 特別適合在閱讀教材後進一步查閱官方技術文件與實作範例。

---

### 🧠 OpenAI 相關文件

- [**OpenAI API Image Generation 文檔**](https://platform.openai.com/docs/guides/image-generation)  
  說明如何使用 `gpt-image-2` 模型進行 **文字轉圖 (Text-to-Image)**、**圖像編輯 (Image-to-Image)** 與 **Inpainting**。  
  包含請求參數（如 `size`、`quality`、`response_format`）及回傳格式的詳細說明。  

- [**OpenAI API 參考文檔 (API Reference)**](https://platform.openai.com/docs/api-reference/images)  
  列出所有影像生成／編輯端點與可用參數，並提供實際範例程式碼。  
  適合在開發階段查詢 API 請求格式與欄位說明。  

- [**OpenAI Cookbook**](https://cookbook.openai.com/)  
  官方技術範例集（recipes），展示如何整合 OpenAI API 至各種應用場景。  
  其中的 [影像生成範例](https://cookbook.openai.com/examples/generate_images_with_gpt_image)  
  對應到本章節的 `client.images.generate()` 教學。


- model: gpt-image-2
    - size (str): 1024x1024 (square), 1536x1024 (landscape), 1024x1536 (portrait) or auto (default)
    - quality: low, medium, high or auto
    - moderation: auto, low

- gpt-image-2 支援在 size 參數中指定任意解析度，但必須同時符合以下所有限制：

    - 最長邊長度必須小於 3840 像素（px）。
    - 寬度與高度都必須是 16 的倍數。
    - 最長邊與最短邊的比例不得超過 3:1。
    - 總像素數不得超過 8,294,400。
    - 總像素數不得少於 655,360。

# OpenAI 提供的 Prompt 撰寫基礎（Prompting Fundamentals）

以下的 Prompt 撰寫原則適用於 GPT 圖像生成模型。這些原則來自 Alpha 測試期間的反覆觀察，涵蓋了圖像生成、圖片編輯、資訊圖表、廣告、人像、UI Mockup，以及合成（Compositing）等多種工作流程。

---

## 1. 結構與目標（Structure + Goal）

建議以一致的順序撰寫 Prompt，例如：

> **背景／場景 → 主體 → 關鍵細節 → 限制條件**

同時加入圖片的預期用途（例如：廣告、UI Mockup、資訊圖表），以幫助模型理解應採用的**模式（Mode）**以及所需的完成度。

對於較複雜的需求，建議使用簡短的小標題或換行分段，而不是將所有內容寫成一大段文字。

---

## 2. Prompt 格式（Prompt Format）

請選擇**最容易維護**的 Prompt 格式。

以下幾種格式都能有效運作：

- Minimal Prompt（極簡 Prompt）
- Descriptive Paragraph（描述性段落）
- JSON-like Structure（類 JSON 結構）
- Instruction-style Prompt（指令式 Prompt）
- Tag-based Prompt（標籤式 Prompt）

只要**意圖（Intent）**與**限制條件（Constraints）**表達清楚，都能得到良好的效果。

若是用於正式產品（Production System），請優先選擇**容易閱讀與維護的 Prompt Template**，而不是追求複雜或巧妙的 Prompt 語法。

---

## 3. 明確描述與品質提示（Specificity + Quality Cues）

盡可能具體描述：

- 材質（Materials）
- 形狀（Shapes）
- 紋理（Textures）
- 視覺媒介（Visual Medium）

例如：

- Photo（照片）
- Watercolor（水彩）
- 3D Render（3D 渲染）

只有在需要時，再加入有針對性的**品質提示（Quality Levers）**，例如：

- Film Grain（底片顆粒）
- Textured Brushstrokes（具有紋理的筆觸）
- Macro Detail（微距細節）

若希望產生**擬真照片（Photorealistic）**，請直接在 Prompt 中加入 **`photorealistic`** 一詞，以強烈啟用模型的擬真模式。

其他類似的描述也有幫助，例如：

- `real photograph`
- `taken on a real camera`
- `professional photography`
- `iPhone photo`

不過，詳細的相機規格（例如焦距、光圈）通常只會被模型**大致參考**，而不會精確模擬真實攝影機的物理效果，因此建議主要用來描述整體風格與構圖，而不是期待完全符合真實相機的成像。

---

## 4. 生成速度與畫質（Latency vs. Fidelity）

若你的應用重視：

- 生成速度
- 大量生成

建議先使用：

```python
quality="low"
```

再評估是否已符合需求。

在許多情況下，`low` 品質仍可提供足夠的畫質，同時大幅縮短生成時間。

但若涉及以下情況，建議比較 `medium` 或 `high` 品質後再正式部署：

- 小尺寸文字
- 密集文字資訊
- 詳細資訊圖表
- 人像特寫
- 身分敏感的人物編輯
- 高解析度輸出

---

## 5. 構圖（Composition）

請明確描述畫面的：

### 取景方式（Framing）

例如：

- Close-up（特寫）
- Wide Shot（廣角）
- Top-down（俯視）

### 視角（Perspective / Angle）

例如：

- Eye-level（平視）
- Low-angle（仰視）

### 光線與氛圍（Lighting / Mood）

例如：

- Soft Diffuse Lighting（柔和漫射光）
- Golden Hour（黃金時刻）
- High Contrast（高對比）

若版面配置很重要，也請描述元素的位置，例如：

- Logo 放在右上角
- 主體置中
- 左側保留留白（Negative Space）

若是：

- 超廣角
- 電影感
- 夜景
- 雨景
- 霓虹燈場景

請加入更多關於：

- 場景尺度（Scale）
- 氛圍（Atmosphere）
- 色彩（Color）

的描述，以避免模型過度追求表面擬真，而犧牲原本想要營造的情境。

---

## 6. 人物、姿勢與動作（People, Pose, and Action）

當畫面包含人物時，請描述：

- 人物大小
- 身體構圖
- 視線方向
- 與物體互動方式

例如：

- 「完整顯示全身，包含雙腳。」
- 「孩子相對於桌子的比例。」
- 「低頭看著打開的書，而不是看鏡頭。」
- 「雙手自然握住腳踏車把手。」

這些描述有助於改善：

- 人體比例
- 動作合理性
- 視線方向

---

## 7. 限制條件（Constraints：修改與保留）

請明確指出：

- 哪些需要改變？
- 哪些必須保持不變？

例如：

- `no watermark`
- `no extra text`
- `no logos/trademarks`
- `preserve identity`
- `preserve geometry`
- `preserve layout`
- `preserve brand elements`

若是圖片編輯，建議使用：

> **「只修改 X（Change only X），其他全部保持不變（Keep everything else the same）。」**

並且在每一次修改時，都重新列出需要保留的項目，以降低模型逐漸偏離（Drift）的情況。

若希望修改非常精準，也可以額外說明不要改變：

- 飽和度（Saturation）
- 對比（Contrast）
- 版面配置（Layout）
- 箭頭（Arrows）
- 標籤（Labels）
- 攝影角度（Camera Angle）
- 周圍物件（Surrounding Objects）

---

## 8. 圖片中的文字（Text in Images）

若圖片中需要出現文字，請：

- 將文字放入引號（`"..."`）
- 或全部使用大寫（`ALL CAPS`）

並指定：

- 字型（Font）
- 字體大小（Size）
- 顏色（Color）
- 擺放位置（Placement）

若遇到：

- 品牌名稱
- 生僻字
- 特殊拼法

建議逐字拼寫（Letter-by-letter），以提高文字生成的正確率。

若圖片包含：

- 小字
- 大量資訊
- 多種字體

建議使用：

- `quality="medium"`
- `quality="high"`

---

## 9. 多張圖片輸入（Multi-image Inputs）

若同時提供多張圖片，請依序標示，例如：

```text
Image 1：產品照片
Image 2：風格參考圖
```

並清楚描述它們之間的關係，例如：

> 將 Image 2 的風格套用到 Image 1。

若需要圖片合成（Compositing），請明確指出元素如何移動，例如：

> 將 Image 1 的鳥放到 Image 2 的大象背上。

---

## 10. 逐步修改，而不是一次加入所有需求（Iterate Instead of Overloading）

雖然 GPT 可以理解很長的 Prompt，但實務上：

先建立一個乾淨的基礎 Prompt，再逐步修改，通常更容易除錯。

例如依序修改：

- Make lighting warmer（光線暖一點）
- Remove the extra tree（移除多餘的樹）
- Restore the original background（恢復原本背景）

也可以利用上下文，例如：

- `same style as before`
- `the subject`

來延續先前的設定。

但若發現模型開始逐漸偏離原始需求（Drift），請重新明確描述那些最重要的細節，以維持生成結果的一致性。

In [ ]:
import os

os.chdir("../../")

範例

In [ ]:
import base64

from openai import OpenAI
from IPython.display import display, HTML

from initialization import credential_init

credential_init()

client = OpenAI()

In [ ]:
prompt = ("A Sumi-e style watercolor painting of mountains during sunset. The sky is depicted with bold "
          "splashes of orange, pink, and purple hues, blending and overlapping in a dynamic composition. "
          "The mountains are represented with expressive brushstrokes, emphasizing their majestic and serene "
          "presence. The focus is on capturing the essence and mood of the scene rather than detailed realism. "
          "The overall effect is serene and contemplative, with a harmonious balance of color and form.")

response = client.images.generate(
    model="gpt-image-2",
    prompt=prompt,
    size="1024x1024",
    quality='medium',
    n=1,
)

image_base64 = response.data[0].b64_json

# 將返回的 base64字串轉換為圖像並且儲存
HTML(f'<img src="data:image/png;base64,{image_base64}"/>')

## 挑戰：如何有效地撰寫 Text-to-Image 提示詞：

> 🎯 **本章學完你將能學會什麼：**
> - 分辨兩種提示詞風格：**標籤式提示 (Danbooru Tags)** 與 **自然語言提示 (Natural Language Prompts)**  
> - 理解不同模型對標籤的支援度差異  
> - 學習如何結合藝術、攝影用語提升生成品質   


在使用 AI 生成圖像（例如 OpenAI 的 Image-2）時，提示詞（prompt）的寫法對結果有決定性影響。主要有兩種提示詞格式：

- 標籤式提示（Danbooru Tag):

    - 範例:    

        masterpiece, best quality, beautiful eyes, clear eyes, detailed eyes, Blue-eyes, 1girl, 20_old, full-body, break, smoking, break, high_color, blue-hair, beauty, black-boots,break, break, Flat vector art, Colorful art, white_shirt, simple_background, blue_background, Ink art, peeking out upper body, Eyes


    - 特點與注意事項：

        - 生效與否取決於模型，不同模型對同一個標籤的理解可能不同。
        - 某些標籤是通用的，例如 1girl、ulzzang，但呈現效果可能差異很大。
        - 一些標籤需要專業知識，例如 chiaroscuro（明暗對照法）。
        - 需要多次嘗試與微調，才能找到最佳組合。

2. 自然語言提示（Natural Language Prompt):

    - 範例:

       A Japanese idol with a breathtakingly glamorous ulzzang appearance,  She has a slim, v-shaped face with large, almond-shaped eyes that sparkle with a lustrous, captivating charm, exuding an aura of youth and ethereal beauty. Her expression is innocent yet alluring, with flawless porcelain skin that enhances her delicate, anime-inspired features. The setting is carefully crafted to complement her enchantment, with soft, diffused lighting that accentuates her mesmerizing, glamorous presence, creating a dreamy and youthful, anime-like allure.


    - 特點與注意事項：

        - 句子寫得流暢、語言優美，能提升生成圖像的質感。

        - 對非母語使用者來說，整合多個描述性詞彙是一大挑戰。

        - 部分詞彙在監控嚴格的模型下可能會被屏蔽，例如 serafuku。

        - Image-2 等模型可能會對過於明顯的 NSFW 提示詞進行攔截。若想生成 NSFW 的內容，建議可以參考開源社群，例如 TensorArt/TensorHub。

## 融入 LCEL 與 LangChain — 讓模型幫你生成更好的 Prompt

> 🎯 **本章學完你將能學會什麼：**
> - 能將 GPT 模型生成的文字提示直接導入影像生成 API  
> - 實作一個自動從描述文字 → prompt → 圖像的完整流程  

本節將介紹如何運用 LangChain 建立一個能自動化撰寫提示詞的流程，並串接影像生成 API，達成端到端的自動圖像生成。

### Step1

可以給予內容，並且讓文字模型幫忙寫提示詞。並且可以考慮使用mlflow監視產出的提示詞

In [ ]:
from textwrap import dedent

from langchain_ollama import ChatOllama
from langchain_core.prompts.image import ImagePromptTemplate
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser


def build_standard_chat_prompt_template(kwargs):
    messages = []

    if 'system' in kwargs:
        content = kwargs.get('system')

        # allow list of prompts for multimodal
        if isinstance(content, list):
            prompts = [PromptTemplate(**c) for c in content]
        else:
            prompts = [PromptTemplate(**content)]

        message = SystemMessagePromptTemplate(prompt=prompts)
        messages.append(message)

    if 'human' in kwargs:
        content = kwargs.get('human')

        # allow list of prompts for multimodal
        if isinstance(content, list):
            prompts = []
            for c in content:
                if c.get("type") == "image":
                    prompts.append(ImagePromptTemplate(**c))
                else:
                    prompts.append(PromptTemplate(**c))
        else:
            if content.get("type") == "image":
                prompts = [ImagePromptTemplate(**content)]
            else:
                prompts = [PromptTemplate(**content)]

        message = HumanMessagePromptTemplate(prompt=prompts)
        messages.append(message)

    chat_prompt_template = ChatPromptTemplate.from_messages(messages)
    
    return chat_prompt_template


model = ChatOllama(model='deepseek-v4-pro:cloud',
                   base_url='https://ollama.com',
                   name='Prompt Generator', temperature=0)

system_template = dedent("""
# ROLE
You are an expert Visual Prompt Engineer specializing in text-to-image generation for GPT-Image-2, GPT-Image-1, and Flux models.

# GOAL
Generate a detailed, descriptive, and technically precise image generation prompt that accurately captures the essence of the provided image description, resulting in a breathtaking masterpiece.

# INPUT
<IMAGE DESC>: A description of an image (provided by the user).

# TASK
Rewrite the input description into a highly detailed, evocative, and technically precise prompt optimized for GPT-Image-2. The output prompt must follow OpenAI's prompting best practices for image generation.

# PROMPT STRUCTURE (follow this order)
Structure the final prompt consistently in this order:
1. Background / Scene — Set the environment and atmosphere.
2. Subject — Describe the main subject(s) clearly.
3. Key Details — Add materials, shapes, textures, visual medium, lighting, mood, color palette.
4. Constraints / Technical Specs — Quality cues, composition, framing, perspective, and restrictions.

For complex requests, use line breaks to separate sections rather than writing one dense paragraph.

# RULES
- Rich Visual Detail: Include style, composition, lighting, mood, subject, color palette, materials, shapes, and textures.
- Avoid Vague Language: Use concrete, specific terms. Never use abstract or ambiguous descriptions.
- Visual Medium: Explicitly state the medium (e.g., photorealistic, watercolor, 3D render, oil painting, digital art, concept art). To trigger photorealistic mode strongly, include the word "photorealistic" or equivalent terms ("real photograph", "taken on a real camera", "professional photography", "iPhone photo").
- Photography & Illustration Expertise: Incorporate relevant techniques (e.g., lens type, perspective, artistic movement, depth of field, bokeh). Note: detailed camera specs (focal length, aperture) are loosely interpreted — use them for overall style and composition guidance only.
- Quality Cues (add only when needed): Include targeted quality levers such as film grain, textured brushstrokes, macro detail, intricate details, hyper-detailed, 8K resolution.
- Composition & Framing: Explicitly describe framing (close-up, wide shot, top-down), perspective/angle (eye-level, low-angle, Dutch angle), and element placement (centered, negative space on left, etc.).
- Lighting & Atmosphere: Specify lighting type (soft diffuse lighting, golden hour, high contrast, cinematic lighting, rim light) and mood.
- People (if applicable): Describe subject size, body composition, gaze direction, interaction with objects, and pose. E.g., "full body visible including feet", "looking down at an open book instead of at the camera", "hands naturally gripping bicycle handlebars".
- Constraints: Explicitly state what to avoid or preserve when relevant: no watermark, no extra text, no logos/trademarks, preserve identity, preserve layout.
- Text in Images: If text must appear in the image, put it in quotes or ALL CAPS, and specify font, size, color, and placement. For rare characters or brand names, spell letter-by-letter.


# CHAIN OF THOUGHT
1. Analyze the input description to identify key elements (subject, setting, style, emotion, intended use).
2. Expand each element with precise artistic and technical details (e.g., "vibrant sunset" → "dramatic golden-hour lighting with long shadows and warm amber hues").
3. Structure the prompt following the Background → Subject → Key Details → Constraints order.
4. Refine the language to be evocative and model-friendly: avoid ambiguity, use descriptive adjectives, ensure intent and constraints are crystal clear.
5. Validate: check completeness, specificity, and that no critical detail from the input is lost.

# OUTPUT
Return ONLY the final optimized prompt. Do not add explanations, markdown formatting, or extra commentary.
""")

human_template = "<IMAGE DESC>: {image_desc}"

input_ = {"system": {"template": system_template},
          "human": {"template": human_template,
                    "input_variable": ["image_desc"]}}
    
chat_prompt_template = build_standard_chat_prompt_template(input_)

nl_prompt_generation_chain = chat_prompt_template | model | StrOutputParser()

### Step2

將生成的提示詞放入影像生成API中

In [ ]:
from operator import itemgetter
from typing import Dict

from langchain_core.runnables import chain, RunnableLambda, RunnableParallel, RunnablePassthrough


@chain
def gpt_image_worker(kwargs: Dict):

    """
    Generates an image using OpenAI's GPT-Image-1 model based on the provided prompt and optional parameters.
    
    Parameters:
    kwargs (Dict): A dictionary containing the following keys:
        - 'nl_prompt' (str): The natural language prompt describing the image to be generated.
        - 'size' (str, optional): The size of the generated image. Default is "1024x1024".
        - 'quality' (str, optional): The quality of the generated image. Default is "medium".
    
    Returns:
    str: image base64 string
    """
    
    print("Start generating image...")
    print(f"prompt: {kwargs['nl_prompt']}")
    client = OpenAI()

    response = client.images.generate(
        model=kwargs.get('model', "gpt-image-2"),
        prompt=kwargs['nl_prompt'],
        size=kwargs.get("size", "1024x1024"),
        quality=kwargs.get('quality', 'medium'),
        moderation=kwargs.get('moderation', 'auto'),
        n=1)

    image_base64 = response.data[0].b64_json
    
    return image_base64


@chain
def base64_to_file(kwargs):

    """
    Save the image from a base64 string
    """
    
    image_base64 = kwargs['image_base64']
    filename = kwargs['filename']
    
    with open(f"{filename}", "wb") as fh:
        fh.write(base64.b64decode(image_base64))

    return image_base64

In [ ]:
import mlflow

mlflow.langchain.autolog()

In [ ]:
# step 1: 生成依照你想要的圖像描述圖像提示詞
step_1 = RunnablePassthrough.assign(nl_prompt=itemgetter('image_desc')|nl_prompt_generation_chain)

# step 2: 生成圖像，並將base64字串放入image_base64變數中
step_2 = RunnablePassthrough.assign(image_base64=gpt_image_worker)

# step 3: 將base64字串儲存為圖像
step_3 = base64_to_file

# 將三個步驟由水管符號(|)連結起來
gpt_image_chain =  step_1|step_2|step_3

In [ ]:
from textwrap import dedent

#人像使用9*16

image_base64_example = gpt_image_chain.invoke({"size": "864x1536",
                     "quality": "medium",
                     "image_desc": dedent("""warhammer 40k, astartes, power armor, chain sword, purity seal, 
                     oil painting, cinematic view, battle field, black templars, sacred light upon the arstartes"""),
                     "filename": "tutorial/week_7/astartes.png"
                    })

In [ ]:
# 將返回的 base64字串轉換為圖像並且儲存
HTML(f'<img src="data:image/png;base64,{image_base64_example}"/>')

In [ ]:
# image_base64_example = gpt_image_chain.invoke({"size": "1536x864",
#                      "quality": "medium",
#                      "image_desc": dedent("""
#                      This is a happy birthday gift card for a 9 years old boy.
#                      He plays chess and is very good at swim.
#                      He excels at mathematics.
#                      """),
#                      "filename": "tutorial/week_7/happy_birthday.png"
#                     })

In [ ]:
,
                     "moderation": "low"image_base64_example = gpt_image_chain.invoke({"size": "864x1536",
                     "quality": "medium",
                     "image_desc": dedent("""A Japanese idol with a breathtakingly glamorous ulzzang appearance, 
                     She has a slim, v-shaped face with large, almond-shaped eyes that sparkle with a lustrous, 
                     captivating charm, exuding an aura of youth and ethereal beauty. Her expression is innocent yet alluring, 
                     with flawless porcelain skin that enhances her delicate, anime-inspired features. 
                     The setting is carefully crafted to complement her enchantment, with soft, diffused lighting that 
                     accentuates her mesmerizing, glamorous presence, creating a dreamy and youthful, anime-like allure.
                     She is cosplaying the Sisters of Battle (Adepta Sororitas)
                     """),
                     "filename": "tutorial/week_7/test_2.png",
                     "moderation": "low"
                    })

In [ ]:
HTML(f'<img src="data:image/png;base64,{image_base64_example}"/>')

# 圖像渲染(Image Render)

「圖像渲染」(Image to Image, 簡稱 Img2Img) 指的是：
在已有圖片的基礎上，搭配新的提示詞 (prompt)，生成另一張風格或內容有所變化的圖片。

## ✨ 特點

1. 輸入與輸出

    - 輸入：一張已有的圖片 + 提示詞

    - 輸出：根據提示詞改造過的圖片

2. 靈活性

    - 可以保留原圖的結構（例如人物姿勢），只改變細節（如髮色、衣服、場景）。

    - 也可以做風格轉換，讓照片變成油畫風、漫畫風、插畫風。

3. 常見應用

    - 修圖：去除背景、修改臉部細節、換衣服。

    - 風格化：將現實照片轉成動漫風、插畫風。

    - 迭代設計：快速嘗試不同的角色服裝、髮型或環境。

    - 局部修改 (Inpainting)：在圖片上指定區域，僅對該區域進行替換或修補。

In [ ]:
# import os

# os.chdir("../../")

In [ ]:
from pathlib import Path
from IPython.display import display, HTML

# Build HTML string
html = '<div style="display: flex; flex-direction: column;">'

html += '<div style="display: flex; justify-content: space-around; margin-bottom: 10px;">'
html += f'''
    <div>
        <img src="Eve_Stellar_Blade.png" style="width: 300px; height: auto; border-radius: 8px; box-shadow: 2px 2px 6px rgba(0,0,0,0.2);" />
    </div>
'''
html += '</div>'

html += '</div>'

# Display the HTML
display(HTML(html))

In [ ]:
from textwrap import dedent

from openai import OpenAI

from initialization import credential_init

credential_init()

client = OpenAI()

prompt = dedent("""
Please rending this image as a realistic photo of a girl cosplaying. A Korean girl with a
slim, v-shaped face with large, almond-shaped eyes that sparkle with captivating charm, exuding 
an aura of youth and ethereal beauty. With flawless skin that enhances her delicate, 
anime-inspired features. The setting is carefully crafted to complement her enchantment, with 
soft, diffused lighting that accentuates her mesmerizing, glamorous presence, creating a dreamy 
and youthful, anime-like allure. Her makeup should resemble the features of K-beauty, such as pale skin tones 
and dewed skin texture. 
""")


image_path = os.path.join("tutorial", "week_7", "Eve_Stellar_Blade.png")

result_edit = client.images.edit(
    model="gpt-image-2",
    image=open(image_path, "rb"), 
    prompt=prompt,
    size="864x1536",
    quality="medium",
)

image_base64 = result_edit.data[0].b64_json

In [ ]:
HTML(f'<img src="data:image/png;base64,{image_base64}" />')

你可以使用一張或是多張圖片做為參照物

- Noshiro 能代 (Azur Lane)

In [ ]:
html = '<div style="display: flex; flex-direction: column;">'

html += '<div style="display: flex; justify-content: space-around; margin-bottom: 10px;">'
html += f'''
    <div>
        <div>
            <img src="Noshiro - Spring.png" style="width: 300px; height: auto; border-radius: 8px; box-shadow: 2px 2px 6px rgba(0,0,0,0.2);" />
            <img src="Noshiro - Summer.png" style="width: 300px; height: auto; border-radius: 8px; box-shadow: 2px 2px 6px rgba(0,0,0,0.2);" />
        </div>
        <div>
            <img src="Noshiro - Fall.png" style="width: 300px; height: auto; border-radius: 8px; box-shadow: 2px 2px 6px rgba(0,0,0,0.2);" />
            <img src="Noshiro - Winter.png" style="width: 300px; height: auto; border-radius: 8px; box-shadow: 2px 2px 6px rgba(0,0,0,0.2);" />
        </div>
    </div>
'''
html += '</div>'

html += '</div>'

# Display the HTML
display(HTML(html))

In [ ]:
image_1 = os.path.join("tutorial", "week_7", "Noshiro - Spring.png")
image_2 = os.path.join("tutorial", "week_7", "Noshiro - Summer.png")
image_3 = os.path.join("tutorial", "week_7", "Noshiro - Fall.png")
image_4 = os.path.join("tutorial", "week_7", "Noshiro - Winter.png")

result_edit = client.images.edit(
    model="gpt-image-2",
    image=[
        open(image_1, "rb"),
        # open(image_2, "rb"),
        # open(image_3, "rb"),
        # open(image_4, "rb"),
    ],
    prompt=dedent("""
    Use the woman from the first image as the model.

Preserve her identity, facial features, hairstyle, expression, skin tone, and overall appearance exactly as in the reference image. 
Do not change the person. She remains the main subject of the advertisement.

Transform the image into a high-end luxury perfume advertisement.

The model is posing as the face of a premium fragrance brand. 
Place an elegant luxury perfume bottle as the hero product in the scene, naturally integrated with the model. 
The bottle is made of flawless crystal glass with refined gold accents, containing warm amber-colored perfume liquid.

Create a sophisticated luxury beauty campaign atmosphere:
- black velvet surface with subtle folds
- soft golden highlights
- elegant cinematic mist around the perfume bottle
- dramatic but flattering lighting
- premium fragrance advertisement aesthetic

The model should maintain the original pose and composition from the first image, while the environment and styling are upgraded into a luxury perfume campaign.

Photography style:
high-end fashion editorial photography, luxury beauty campaign, photorealistic, cinematic lighting, shallow depth of field, premium magazine advertisement quality.

Color palette:
deep black, champagne gold, warm amber, soft skin tones.

The woman's face remains the primary focus.
The perfume bottle is the secondary focal point.
    """),
    size="1024x1536",
    quality="medium"
)

image_base64 = result_edit.data[0].b64_json

In [ ]:
HTML(f'<img src="data:image/png;base64,{image_base64}"/>')

### 將不同圖片的內容融合在一起

圖片來源: https://tensor.art/u/629260971684229814

In [ ]:
html = '<div style="display: flex; flex-direction: column;">'

html += '<div style="display: flex; justify-content: space-around; margin-bottom: 10px;">'
html += f'''
    <div>
        <div>
            <img src="maehara-1.jpg" style="width: 300px; height: auto; border-radius: 8px; box-shadow: 2px 2px 6px rgba(0,0,0,0.2);" />
            <img src="maehara-2.jpg" style="width: 300px; height: auto; border-radius: 8px; box-shadow: 2px 2px 6px rgba(0,0,0,0.2);" />
        </div>
    </div>
'''
html += '</div>'

html += '</div>'

# Display the HTML
display(HTML(html))

In [ ]:
image_a = os.path.join("tutorial", "week_7", "maehara-1.jpg")
image_b = os.path.join("tutorial", "week_7", "maehara-2.jpg")

In [ ]:
result_edit = client.images.edit(
    model="gpt-image-2",
    image=[
        open(image_a, "rb"),
        open(image_b, "rb"),
    ],
    prompt=dedent("""
    Fusion the two images to create a high definition 8k movie poster with the text as the background. 
    """),
    size="1024x1536",
    quality="high"
)

image_base64 = result_edit.data[0].b64_json

In [ ]:
HTML(f'<img src="data:image/png;base64,{image_base64}"/>')

# 教學用投影片內容（正經版）

在觀摩多種不是很正經的範例後，我們來探討一個嚴謹的教學應用案例。

## 圖像生成邏輯流程

本系統根據內容性質自動選擇最適合的視覺化方式：

### 1. LLM 判斷機制
- 輸入一段敘述內容後，由大型語言模型(LLM)進行分析判斷
- 判斷產出的圖片應屬於哪種類型：
  - **概念性插畫** (`GeneralImage`)
  - **精確數學表達** (`MatplotlibImage`)

### 2. 分流處理邏輯
```python
if 判斷為 GeneralImage:
    直接將 image_prompt 送入 GPT-Image-2 API 生成插畫
else:
    根據 image_prompt 產生 matplotlib 作圖代碼
    將代碼送入 GPT-Image-2 API 執行並生成圖表

結構化輸出

In [ ]:
from typing import Literal, Union

from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class GeneralImage(BaseModel):    
    name: Literal["general"] = "general"    
    image_prompt: str = Field(description="Use for conceptual, artistic, or illustrative visualizations (analogies, real-world examples, conceptual flows) that aid intuition and cannot be represented by a plot.")
    
class MatplotlibImage(BaseModel):    
    name: Literal["matplotlib"] = "matplotlib"    
    image_prompt: str = Field(description="Use for precise technical visualizations where the exact relationship between variables is the primary teaching point (function plots, data charts, geometric proofs).")
    
class Output(BaseModel):    
    reasoning: str = Field(description="Explanation of why an image is unnecessary (e.g., textual sufficiency, pure symbolic logic, or avoiding cognitive overload).")    
    image_selection: Union[GeneralImage, MatplotlibImage, None] = Field(description="The chosen image type if necessary, otherwise None.")


output_parser = PydanticOutputParser(pydantic_object=Output)
format_instructions = output_parser.get_format_instructions()

系統提示詞

In [ ]:
from textwrap import dedent


SYSTEM_PROMPT = dedent("""
# ROLE
You are an expert Instructional Designer and Visual Communications Specialist. 

Your goal is to determine if a slide requires a visualization to enhance learning and, if so, which type of image is most appropriate.

# DECISION LOGIC
1. **GeneralImage**: Choose this if the goal is to provide **intuition** via a metaphor, illustration, or conceptual diagram.
2. **MatplotlibImage**: Choose this if the goal is to provide **precision** via a mathematical plot, data graph, or technical chart.
3. **Output (No Image)**: Choose this if the goal is best achieved through **clarity** via concise text, LaTeX equations, or if an image would be redundant/distracting.

etermine if an image is necessary. If it is, specify whether it should be a General illustration or a Matplotlib plot and provide the corresponding prompt/code.

# OUTPUT FORMAT
Return a structured response indicating the choice and the justification based on the pedagogical goals of the slide.""")

第一層的pipeline

兩個輸入: 投影片的文字內容(slide_content)和講稿(speaker_notes, 最後由TTS產生語音)

In [ ]:
input_ = {"system": {"template": SYSTEM_PROMPT},
          "human": {"template": "<Title>: {title}\n\n<Slide Content>: {slide_content}\n\n<Speaker Note>: {speaker_notes}\n\n\nStructured Output Instruction: {format_instructions}",
                    "input_variable": ["slide_content", "speaker_notes"],
                    "partial_variables": {'format_instructions': format_instructions}}}

chat_prompt_template = build_standard_chat_prompt_template(input_)

pipeline = chat_prompt_template | model | output_parser

定義投影片的模板

In [ ]:
image_template = "tutorial/week_7/layout_template.png"
image_template_rb = open(image_template, "rb")
image = [image_template_rb]

投影片的內容: 根據特定條件，由LLM生成

In [ ]:
slide_content = [
        "• Vectors have both magnitude and direction",
        "• Represented as arrows: length = magnitude",
        "• Examples: velocity, force, acceleration",
        "• Scalar: only magnitude (speed, mass, time)"
      ]
speaker_notes = "The key to solving our river puzzle is understanding vectors. Unlike regular numbers that just have size, vectors have both size AND direction. Think of them as arrows - the length tells you how strong, and the direction tells you where it's pointing. Velocity is a vector - 60 km/h north is different from 60 km/h east. Speed is just the magnitude part without direction. This distinction is crucial for 2D motion."

slide_title = "Vectors: Direction Matters"

In [ ]:
output = pipeline.invoke({"slide_content": ",\n".join(slide_content), "speaker_notes": speaker_notes, "title": slide_title})

In [ ]:
output.image_selection

In [ ]:
# 根據邏輯產生的硬編碼
visual_instruction = "Generate a supporting educational diagram based on this description:"

visual_content = output.image_selection.image_prompt

送入Image-2的提示詞

In [ ]:
prompt = dedent(f"""
            Think step by step.
            
            Create a high-quality educational slide image using the provided image as the background template.
            
            STRICT TEMPLATE RULES:
            - Use the provided image as the fixed layout and background
            - Preserve its structure, spacing, and visual hierarchy
            - Do NOT alter the template design
    
            TEXT PLACEMENT:
    
            Title (place in the title area exactly):
            {slide_title}
    
            Bullet Points (place in the content area, left-aligned):
            {chr(10).join([f"• {item}" for item in slide_content])}
    
            TEXT REQUIREMENTS (CRITICAL):
            - Preserve all numbers, equations, and units EXACTLY
            - Do NOT reword, summarize, or simplify
            - Ensure high readability and clean alignment
            - Use consistent bullet spacing
    
            VISUAL CONTENT:
            {visual_instruction}
            {visual_content}
            - Place it in the designated visual area of the template
            - Ensure it does NOT overlap with text
            - Keep proportions clean and uncluttered
    
            LAYOUT RULES:
            - Maintain clear separation between text and visuals
            - Do not overcrowd the slide
            - Adjust diagram scale if needed to fit cleanly
    
            STYLE:
            - Professional educational slide
            - Clean, minimal, high contrast
            - No unnecessary decorations
            
            The final result must look like a polished presentation slide rendered on the provided template.
            """)

In [ ]:
result_edit = client.images.edit(
                    model="gpt-image-2",
                    image=image,
                    prompt=prompt,
                    quality="medium"
                )

In [ ]:
image_base64 = result_edit.data[0].b64_json

HTML(f'<img src="data:image/png;base64,{image_base64}"/>')

需要用matplotlib支援的範例

In [ ]:
slide_content = [
        "Vector u = 2i - 3j",
        "Write in component form",
        "Calculate magnitude",
        "Find direction angle",
        "Sketch geometric representation"
      ]

speaker_notes = "Let's put it all together. Given vector u equals 2i minus 3j, can you convert this to component form, find its magnitude and direction, and sketch what it looks like? Think about what the negative sign means for the j component."

slide_title = "Vector Representation Mastery"

output = pipeline.invoke({"slide_content": ",\n".join(slide_content), "speaker_notes": speaker_notes, "title": slide_title})

In [ ]:
output.image_selection

將prompt渲染成代碼

代碼生成pipeline:
- 結構化輸出 

In [ ]:
class MatplotlibCodeResponse(BaseModel):
    """Pydantic model for structured matplotlib code finetuning response."""
    
    code: str = Field(description="The complete Python matplotlib code")

系統提示詞

In [ ]:
MATPLOTLIB_SYSTEM_PROMPT = dedent("""
You are an expert Python visualization engineer specializing in matplotlib rendering for pedagogical content. Your goal is to produce clean, robust, and visually clear code that helps students understand the underlying concepts.

--------------------------------
TASK & PEDAGOGICAL GUIDANCE
--------------------------------
Generate Python matplotlib code based on:
<PROMPT>: The image prompt

Your visual should:
- Prioritize Clarity: Avoid complexity or fancy decorations. A simple, clean diagram that clarifies a core concept is most effective.
- Manage Cognitive Load: Focus only on essential elements that improve understanding.

--------------------------------
CORE RULES
--------------------------------
1. Output ONLY valid Python code. No markdown, no explanations.
2. Code must run as-is. Include all required imports (matplotlib.pyplot, numpy, textwrap, etc.).
3. Save output to `output.png` using `bbox_inches="tight"`, `dpi=200`.
4. Prevent overlapping elements using `plt.tight_layout()` or precise coordinate calculations.
5. LaTeX: ALL math expressions MUST use LaTeX format via raw strings: r"$ ... $".
6. Precision: Use scipy.stats for precise statistical values; do not hardcode approximations.
7. Robustness: Handle variable content lengths gracefully to prevent text overflow.

--------------------------------
DATA RANGE & CENTERING RULES
--------------------------------
8. Dynamic Axis Scaling: Ensure the main content (e.g., data points, regression lines, focal points) is centrally positioned and well-framed.
9. Adaptive Limits: Calculate plot limits (`plt.xlim`, `plt.ylim`) based on the actual data range to avoid excessive whitespace or cutting off important features.
10. Margin Management: Apply a reasonable padding (e.g., 5-10%) around the data extremes to ensure a balanced and professional visual composition.

--------------------------------
CONSTRAINTS
--------------------------------
- Use appropriate figure sizes and normalized coordinates for text-based layouts.
- Maintain a clean, minimal style: avoid excessive colors, decorative fonts, or complexity.
- Ensure the output is stable, visually balanced, and mathematically precise.
""")

建立pipeline

In [ ]:
output_parser = PydanticOutputParser(pydantic_object=MatplotlibCodeResponse)
format_instructions = output_parser.get_format_instructions()

human_template = dedent("""\
<PROMPT>: {prompt}

Output Format Instruction: {format_instructions}
""")

text_prompt_template = {"template": human_template, 
                        "input_variables": ["prompt"],
                        "partial_variables": {'format_instructions': format_instructions}}

input_ = {
    "system": {"template": MATPLOTLIB_SYSTEM_PROMPT},
    "human": [text_prompt_template],
}

chat_prompt_template = build_standard_chat_prompt_template(input_)

model_coding = ChatOllama(model="deepseek-v4-pro:cloud", temperature=0,
                          base_url='https://ollama.com', name='coder')

pipeline_coding = chat_prompt_template | model_coding | output_parser

In [ ]:
output_code = pipeline_coding.invoke({"prompt": output.image_selection.image_prompt})
visual_content = output_code.code

In [ ]:
print(visual_content)

In [ ]:
exec(visual_content)

最後送入Image-2的prompt

In [ ]:
from copy import copy

In [ ]:
visual_instruction = "Render a precise mathematical plot with the second image:\n- Prioritize the accuracy of the graph, like copy-paste\n"

bullets = chr(10).join(f"{item}" for item in slide_content)

prompt = dedent(f"""
        STRICT TEMPLATE RULES:
        - Use the provided image as the fixed layout and background
        - Preserve its structure, spacing, and visual hierarchy
        - Do NOT alter the template design

        TEXT PLACEMENT:

        Title (place in the title area exactly):
        {slide_title}

        Bullet Points (place in the content area, left-aligned):
        {bullets}

        TEXT REQUIREMENTS (CRITICAL):
        - Preserve all numbers, equations, and units EXACTLY
        - IMPORTANT OVERRIDE: $...$ delimiters are LaTeX markup, NOT slide content. Strip the $ symbols from the final image entirely. Render only the enclosed expression as properly formatted mathematical notation (italic variables, upright numbers, proper spacing).
        - Do NOT reword, summarize, or simplify
        - Ensure high readability and clean alignment
        - Use consistent bullet spacing
        - Content must be appropriate for AP-aligned coursework (academically rigorous and precise)

        VISUAL CONTENT:
        {visual_instruction}
        - Place it in the designated visual area of the template
        - Ensure it does NOT overlap with text
        - Keep proportions clean and uncluttered

        LAYOUT RULES:
        - Maintain clear separation between text and visuals
        - Do not overcrowd the slide
        - Adjust diagram scale if needed to fit cleanly

        STYLE:
        - Professional educational slide
        - Clean, minimal, high contrast
        - No unnecessary decorations
        - Designed for AP curriculum presentation standards
        - Color coding rule: Blue is used exclusively for key concepts (important terms and core ideas)
        - Key formulas appear in a subtle box or highlight -- exam-ready emphasis

        The final result must look like a polished, academically rigorous AP presentation slide rendered on the provided template.
    """)

with open("output.png", "rb") as f:
    image_input = copy(image)
    
    image_input.append(f)
    
    result_edit = client.images.edit(
                        model="gpt-image-2",
                        image=image_input,
                        prompt=prompt,
                        quality="medium"
                    )

In [ ]:
image_base64 = result_edit.data[0].b64_json

HTML(f'<img src="data:image/png;base64,{image_base64}"/>')

In [ ]:
import aiofiles
import base64

output_filename = "tutorial/week_7/matplotlib_image_example.png"

async with aiofiles.open(output_filename, "wb") as fh:
    await fh.write(base64.b64decode(image_base64))

一隻AI端到端生成的教學影片

所有的技巧你都學過: 
- LLM進行內容生成
- TTS將講稿內容變成聲音
- 使用OpenAI GPT-Image-2進行圖片生成。

剩下的就是想辦法串起來

In [ ]:
from IPython.display import Video

from IPython.display import HTML

HTML("""
<video width="600" controls>
  <source src="final_presentation_celery_1768_1776927683.mp4" type="video/mp4">
</video>
""")

一個5分鐘左右，整體生成時間<30分鐘，由10張投影片所組成的教學影片成本大概是0.5-1美金

## 局部修補 (Inpaint)

你可以提供一個遮罩 (mask) 來指定圖像中要被編輯的區域。

當在 GPT Image 中使用遮罩時，額外的指令會一併傳送給模型，以便更好地引導編輯過程。

### 遮罩的要求

要編輯的圖片與遮罩必須為相同的格式與尺寸，且檔案大小需小於 50MB。

遮罩圖片必須包含 Alpha 通道。如果你是使用圖像編輯工具來製作遮罩，請確保在儲存時保留 Alpha 通道。


### Bug Report

https://community.openai.com/t/gpt-image-1-problems-with-mask-edits/1240639/15

Image-1 在 inpainting 似乎做的很糟糕。

In [ ]:
html = '<div style="display: flex; flex-direction: column;">'

html += '<div style="display: flex; justify-content: space-around; margin-bottom: 10px;">'
html += f'''
    <div>
        <div>
            <img src="Noshiro - Winte - Mask.png" style="width: 300px; height: auto; border-radius: 8px; box-shadow: 2px 2px 6px rgba(0,0,0,0.2);" />
        </div>
    </div>
'''
html += '</div>'

html += '</div>'

# Display the HTML
display(HTML(html))

In [ ]:
image_in = os.path.join("tutorial", "week_7", "Noshiro - Winter.png")
image_mask = os.path.join("tutorial", "week_7", "Noshiro - Winte - Mask.png")

result_edit = client.images.edit(
    model="gpt-image-2",
    image=open(image_in, "rb"),
    mask=open(image_mask, "rb"),
    prompt=dedent("""
    In the winter, a girl walking on water and holding Mjölnir. Mjölnir is surrounded with electricity and current. 
    """),
    size="1024x1536",
    quality="medium"
)

image_base64 = result_edit.data[0].b64_json

In [ ]:
HTML(f'<img src="data:image/png;base64,{image_base64}"/>')

Image-2 在 Mask上延續著從Image-1就有的問題。

# Agent（代理型系統）

> 🎯 **本章學完你將能學會什麼：**
> - 理解 **Agent** 的核心概念與與傳統 LLM 回答的差異  
> - 掌握 **ReAct (Reasoning + Acting)** 的行動思考循環架構  
> - 學會以 LangChain 建立具邏輯推理與工具調用能力的 Agent  
> - 熟悉如何用 **MLflow** 監控模型執行過程與日誌  
> - 實作能自行規劃步驟並計算問題的智能代理  

Anthropic definition for Agent: `LLMs autonomously using tools in a loop`

## ReAct Framework

本節將說明 ReAct 的概念與執行結構，幫助你理解思考（Reasoning）與行動（Acting）如何交互作用。

- ReAct: Reasoning - Action

- ReAct Agent 的運作流程大致是：

    1. 思考 (Reasoning)：根據當前的上下文，生成內部的推理或計劃。

    2. 行動 (Acting)：根據推理的結果，決定要採取的動作（例如查詢工具、呼叫 API、檢索知識）。

    3. 觀察 (Observation)：得到工具或環境回饋。

    4. 迭代：將觀察結果再輸入回去，進入下一輪思考。

    直到：

    a. 達到最終答案，或

    b. 遇到設置的停止條件（例如 token 限制、步數限制、明確的 "結束" 信號）。

In [ ]:
from IPython.display import HTML

html = '<div style="display: flex; flex-direction: column;">'

html += '<div style="display: flex; justify-content: space-around; margin-bottom: 10px;">'
html += f'''
    <div>
        <div>
            <img src="agent_diagram.png" style="width: 1200px; height: auto; border-radius: 8px; box-shadow: 2px 2px 6px rgba(0,0,0,0.2);" />
        </div>
    </div>
'''
html += '</div>'

html += '</div>'

# Display the HTML
display(HTML(html))

In [ ]:
import os

os.chdir("../../")

In [ ]:
import mlflow
from langchain_community.callbacks import MlflowCallbackHandler

from initialization import credential_init

credential_init()

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model='deepseek-v4-pro:cloud',
                     base_url='https://ollama.com',
                     name='agent model', temperature=0)

建立ChatPromptTemplate的工具

In [ ]:
from langchain_core.prompts.image import ImagePromptTemplate
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate, PromptTemplate


def build_standard_chat_prompt_template(kwargs):
    messages = []

    if 'system' in kwargs:
        content = kwargs.get('system')

        # allow list of prompts for multimodal
        if isinstance(content, list):
            prompts = [PromptTemplate(**c) for c in content]
        else:
            prompts = [PromptTemplate(**content)]

        message = SystemMessagePromptTemplate(prompt=prompts)
        messages.append(message)

    if 'human' in kwargs:
        content = kwargs.get('human')

        # allow list of prompts for multimodal
        if isinstance(content, list):
            prompts = []
            for c in content:
                if c.get("type") == "image":
                    prompts.append(ImagePromptTemplate(**c))
                else:
                    prompts.append(PromptTemplate(**c))
        else:
            if content.get("type") == "image":
                prompts = [ImagePromptTemplate(**content)]
            else:
                prompts = [PromptTemplate(**content)]

        message = HumanMessagePromptTemplate(prompt=prompts)
        messages.append(message)

    chat_prompt_template = ChatPromptTemplate.from_messages(messages)

    return chat_prompt_template

- Langchain 1.0
- 最簡單的呼叫，但沒太大的用處。重點還是要定義工具

In [ ]:
import mlflow
from langchain.agents import create_agent
# from langchain_community.callbacks.mlflow_callback import MlflowCallbackHandler

# mlflow.langchain.autolog()

agent = create_agent(
    model=model,
    name="Simple Agent"
)

response = agent.invoke({"messages": "介紹Langchain。使用繁體中文"})

In [ ]:
print(response['messages'][-1].content)



像 ChatGPT 一樣，一個 token 一個 token 輸出。## Streaming

### Token Streaming

像 ChatGPT 一樣，一個 token 一個 token 輸出。

In [ ]:
for chunk in agent.stream(
    {
        "messages": [
            {"role": "user", "content": "介紹Langchain。使用繁體中文"}
        ]
    },
    stream_mode="messages",
):
    print(chunk[0].content.strip(" "), end="")

### Update Streaming

In [ ]:
for update in agent.stream(
    {"messages": {"role": "user", "content": "介紹Langchain。使用繁體中文"}},
    stream_mode="updates",
):
    print(update)

## Agent 範例一

讓Agent使用工具來回答問題

In [ ]:
from textwrap import dedent

from langchain_core.runnables import Runnable
from langchain.tools import BaseTool
from langchain_core.output_parsers import StrOutputParser


SOLUTION_TOOL_SYSTEM_PROMPT = dedent("""\
你是一位在國際關係、地緣政治與全球產業分析領域的資深專家。

你的任務是根據所提供的問題，產出嚴謹、結構化且具可行性的分析。

指導原則：

1. 分析嚴謹性
   - 你的推理應建立在既有的經濟、政治與產業分析框架之上。
   - 當資料不確定或屬於情境推演時，需明確說明假設條件。

2. 結構化輸出
   - 使用清楚的段落與邏輯架構來組織內容。
   - 在適當情況下，應包含：
     - 背景脈絡
     - 關鍵驅動因素與限制條件
     - 利害關係人或國家層級分析
     - 風險與不確定性
     - 策略選項或解決方案

3. 解決導向思維
   - 不僅描述問題，還需提出具現實可行性的解決方案或政策選項。
   - 評估不同方案之間的權衡取捨。

4. 中立性與精確性
   - 避免意識形態偏見。
   - 未經充分依據不得進行推測。
   - 使用精確且專業的語言。

你不負責工具管理或對話流程控制。
除非缺乏關鍵資訊，否則不主動提出追問。
你的唯一職責是產出高品質的分析與解決方案。

請以繁體中文作答。
""")

In [ ]:
class SolutionTool(BaseTool):
    name:str = "SolutionTool" 
    description:str = dedent("""\
一個專門設計用於產出嚴謹、以證據為基礎分析與可執行解決方案的分析工具。

當出現以下情況時應使用此工具：
- 使用者要求策略分析
- 問題涉及國家、地緣政治或產業
- 需要結構化且有證據基礎的回答

此工具負責：
- 進行深入的地緣政治、國際關係與全球產業分析。
- 評估國家層級策略、產業競爭力、供應鏈，以及政策影響。
- 產出結構化分析結果，例如：
  - 根本原因分析
  - 情境比較
  - 風險與機會評估
  - 策略建議與解決方案選項

輸入內容應清楚說明：
- 需要分析的問題或議題
- 相關的國家、地區或產業
- 決策或策略目標（例如：政策設計、投資、風險緩解）

此工具專注於分析嚴謹性與解決方案品質，而非對話管理或任務編排。
""")

    runnable: Runnable

    @classmethod
    def create(cls, llm: Runnable):

        input_ = {"system": {"template": SOLUTION_TOOL_SYSTEM_PROMPT},
                  "human": {"template": "{user_query}",
                            "input_variable": ["user_query"]}}

        chat_prompt_template = build_standard_chat_prompt_template(input_)

        pipeline = chat_prompt_template | llm | StrOutputParser()

        return cls(runnable=pipeline)
    
    def _run(self, query: str):
        
        return  self.runnable.invoke({"user_query": query})
    
    def _arun(self, query: str):
        raise NotImplementedError("This tool does not support async")

**Agent 與 Tool 調用策略簡要說明**

在使用 agent 與工具（tool）架構時，即使工具本身已包含 description，仍建議在 agent 的 system prompt 中明確定義「何時應該調用工具」。原因在於，模型在實務上通常不會穩定依賴 tool description 來做決策，且 system prompt 的優先權高於 tool description，因此更能影響行為。

工具的 description 主要用來說明「工具能做什麼」，例如其分析能力與輸出形式；但 agent 真正需要的是「何時必須使用該工具」，這屬於決策層的規則，應在 system prompt 中定義。

若沒有明確規範，模型可能會直接回答問題，而不調用工具，即使工具能提供更高品質的結果。因此建議在 system prompt 中加入清楚的使用條件，例如：當問題涉及策略分析、地緣政治、跨國議題或需要結構化分析時，必須調用特定工具，且不應直接作答。

一個有效的設計方式是將 agent 作為決策層，負責判斷是否調用工具；而工具則作為分析引擎，專注於產出高品質結果。為了提升穩定性，也可加入額外規則，例如：即使模型能自行回答，只要工具能提供更佳結果，仍必須優先使用工具。

總結而言，tool description 與 system prompt 扮演不同角色：前者定義能力，後者規範行為。兩者需搭配設計，才能確保 agent 正確且穩定地調用工具。


In [ ]:
AGENT_SYSTEM_PROMPT = dedent("""\
你是一位負責高風險分析任務的資深協調（orchestration）代理人。

請嚴格且明確地遵循以下流程：

1. PLAN（規劃）
   - 拆解問題。
   - 定義分析目標與成功判準。
   - 此階段可為隱性，但必須指導後續所有步驟。

2. ACTION（行動）
   - 在適當情況下，將深入分析任務委派給 SolutionTool。
   - 每一次工具調用必須具備單一且明確的目的。

   【SolutionTool 使用規則（強制）】
   在以下情況下，你必須使用 SolutionTool：
   - 使用者要求策略性分析、地緣政治或產業分析
   - 問題涉及多個國家、政策、供應鏈或全球性動態
   - 任務需要結構化、基於證據或比較性的分析
   - 問題具有高複雜度或高決策影響

   規則補充：
   - 即使你可以自行回答，但若 SolutionTool 能提供更高品質結果，仍必須使用該工具
   - 不得在上述情況下直接作答，必須進行工具調用

3. SYNTHESIS（綜合）
   - 將分析結果整合為結構化結論。
   - 本段落必須標示為 "SYNTHESIS"。

4. CRITIQUE（批判）
   - 此為必須輸出的段落。
   - 指出關鍵假設。
   - 強調主要不確定性與潛在失效模式。
   - 說明信心水準（低 / 中 / 高）。
   - 本段不得引入新的分析內容。

最終輸出格式（強制）：

### SYNTHESIS
<內容>

### CRITIQUE
- Assumptions:
- Uncertainties:
- Failure modes:
- Confidence level:

規則：
- 若缺少 CRITIQUE，則任務視為失敗。
- 不得讓工具單獨決定最終答案。
- 維持分析上的中立性。
- 使用繁體中文回復
""")

In [ ]:
solution_model = ChatOllama(model='deepseek-v4-pro:cloud',
                            base_url='https://ollama.com',
                            name='solution model', temperature=0)

tools = [SolutionTool.create(llm=solution_model)]

# ** 別調 REASONING=TRUE **，工具會跑不出來
model = ChatOllama(model='deepseek-v4-pro:cloud',
                   base_url='https://ollama.com',
                   name='Agent Model', temperature=0)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=AGENT_SYSTEM_PROMPT,
    name = "analysis_agent"
)

測試Agent是否可以看到工具

In [ ]:
agent.invoke({"messages": "你有那些工具可以使用?"})

來試試看比較有趣的問題，雖然可能已經過期了:

In [ ]:
content = dedent("""\

請根據以下台美關稅與投資談判內容，分析對台灣的產業、社會與國際外交影響，並提出可能的風險與機會。請使用繁體中文回答，條理清楚，分點呈現，並保持分析深度：

### 已知內容：
1. 美國同意將對台灣出口商品的對等關稅降至 15%，比原本暫時性實施的 20% 暫定稅率更低。
2. 此 15% 的稅率不會再疊加（即不疊加現有最惠國稅率等），等同於美國對日本、韓國等主要貿易夥伴的等級。
3. 台灣的半導體和相關高科技產品在美國《232條款》下獲得「最優惠待遇」（Most Favored Nation treatment）。
4. 台灣企業（尤其是半導體、ICT、人工智慧等產業）承諾將在美國進行大規模投資，官方報導與美方公告提及至少美金數千億美元的投資承諾，包括直接擴廠與信用保證措施。
5. 美國商務部長盧特尼克在公開訪談中提到，美國政府的一個長期目標是讓全球約 40% 的台灣半導體供應鏈產能轉移到美國，以提升美國在先進晶片製造的自給自足程度並強化國家安全。
   - 在台美目前宣布的貿易協議與投資備忘錄（MOU）中，沒有具體規定企業必須把 40% 半導體產能搬到美國。協議主要內容是：台灣科技企業承諾將對美投資巨額資金（直接投資與信用保證合計約 5,000 億美元），作為交換美方將對台關稅調降並在特定情況下豁免《232條款》課徵的高額關稅。換句話說，40% 產能移轉是一個美方政策目標與談判籌碼，但目前沒有法律或協議文本要求台灣企業達成該具體比例。

### 分析要求：
1. **產業面**：分析對台灣半導體、高科技、ICT、AI等產業的影響，包括成本、供應鏈、競爭力與產業結構調整的風險與機會。
2. **社會面**：分析對台灣就業、市場薪資、社會輿論與政治角力的可能影響。
3. **國際外交面**：分析台美協議對台灣地緣政治位置、與其他主要貿易夥伴關係及區域產業競爭格局的影響。
4. 提出明確的 **風險與機會**。
5. 條理清楚、分點呈現，保持分析深度。
""")

In [ ]:
for update in agent.stream(
    {"messages": {"role": "user", "content": content}},
    stream_mode="updates",
):
    print(update)

In [ ]:
print(update['model']['messages'][0].content)

In [ ]:
# 試試看不同的model給的分析
solution_model = ChatOllama(model='gemma4:31b-cloud',
                            base_url='https://ollama.com',
                            name='solution model', temperature=0)

tools = [SolutionTool.create(llm=solution_model)]

model = ChatOllama(model='deepseek-v4-pro:cloud',
                   base_url='https://ollama.com',
                   name='Agent Model', temperature=0)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=AGENT_SYSTEM_PROMPT,
    name = "analysis_agent"
)

In [ ]:
for update in agent.stream(
    {"messages": {"role": "user", "content": content}},
    stream_mode="updates",
):
    print(update)

In [ ]:
print(update['model']['messages'][0].content)

我們剛剛的Agent只包含了一個工具: SolutionTool。我們可以能加入另一個工具，來幫助Agent對於自己的答案進行反思

In [ ]:
CRITIC_TOOL_SYSTEM_PROMPT = dedent("""\
你是一位批判性審查代理人，扮演獨立的紅隊（red-team）分析角色。

你的唯一職責是對既有分析進行批判（CRITIQUE）。

嚴格限制：
- 你不得引入任何新的事實、論點或分析。
- 你的批判必須完全基於所提供的綜合內容（synthesis）。
- 你不得嘗試修正或重寫該分析。

你的任務是識別分析中的弱點、風險與不確定性。

請聚焦於以下面向：
1. 隱含或未明示的假設
2. 關鍵不確定性與未知因素
3. 邏輯漏洞或過度自信
4. 當假設失效時的潛在失敗模式

你必須產出以下結構化格式：

### CRITIQUE
- Assumptions:
- Uncertainties:
- Failure modes:
- Confidence level（低 / 中 / 高）

規則：
- 若無法找到弱點，必須明確說明。
- 不得為了禮貌而弱化批評。
- 不得提出解決方案。
- 不得重複綜合內容（synthesis）。

請以繁體中文作答。
""")

In [ ]:
class CriticTool(BaseTool):
    name: str = "CriticTool"
    description: str = dedent("""\
    對既有的綜合分析進行批判性審查。

    在不引入新的分析內容的前提下，識別其中的假設、不確定性以及潛在的失效模式。
    """)

    runnable: Runnable

    @classmethod
    def create(cls, llm: Runnable):

        input_ = {"system": {"template": CRITIC_TOOL_SYSTEM_PROMPT},
                  "human": {"template": "{user_query}",
                            "input_variable": ["user_query"]}}

        chat_prompt_template = build_standard_chat_prompt_template(input_)

        pipeline = chat_prompt_template | llm | StrOutputParser()

        return cls(runnable=pipeline)

    def _run(self, query: str):
        
        return  self.runnable.invoke({"user_query": query})
    
    def _arun(self, synthesis: str):
        raise NotImplementedError("This tool does not support async")

In [ ]:
 critic_model = ChatOllama(model='gpt-oss:120b-cloud',
                            base_url='https://ollama.com',
                            name='critic model', temperature=0)

tools = [SolutionTool.create(llm=solution_model), CriticTool.create(llm=critic_model)]

現在Agent要做的事情是透過 (Solution -> Critic) x n 的過程來產生一個答案。

我通常的作法是將 Tools 先寫好，然後說明Agent需要做到的事情，讓LLM來產生AGENT_SYSTEM_RPOMPT

In [ ]:
AGENT_SYSTEM_PROMPT = dedent("""\
你是一個進階的分析型 ReAct Agent，專門用於產出高品質、具證據基礎且經過批判性檢驗的答案。

你可以使用以下工具：
- SolutionTool：用於產出結構化、嚴謹的分析與解決方案
- CriticTool：用於對既有分析進行批判性審查（只能指出假設、不確定性與潛在失效模式，不可引入新內容）

---

## 🎯 任務目標

1. 完整理解使用者問題
2. 在行動前先制定清晰計畫
3. 透過工具反覆迭代提升答案品質
4. 對中間結果進行批判性評估
5. 持續優化直到答案具備決策價值
6. 最終整合為完整且一致的回答

---

## 🧠 推理流程（ReAct + 規劃 + 迭代）

你必須嚴格遵循以下循環流程：

---

### Step 1 — PLAN（規劃）
- 拆解問題
- 識別：
  - 需要分析的核心問題
  - 存在的不確定性
  - 應優先使用的工具
- 制定清晰的行動計畫

格式：
PLAN:
- Step 1: ...
- Step 2: ...

---

### Step 2 — ACT（執行工具）
- 在以下情況使用 SolutionTool：
  - 涉及策略、地緣政治、產業或需要結構化分析
- 在以下情況使用 CriticTool：
  - 已有分析結果，需要進行壓力測試或批判檢驗

格式：
ACTION:
Tool: <工具名稱>
Input: <輸入內容>

---

### Step 3 — OBSERVATION（觀察）
- 記錄工具回傳結果

格式：
OBSERVATION:
<工具輸出內容>

---

### Step 4 — THOUGHT（推理 / Chain-of-Thought）
- 評估：
  - 當前答案是否足夠？
  - 存在哪些弱點？
  - 是否需要再次使用工具？

重要規則：
- 必須進行逐步推理
- 推理需精簡但具邏輯
- 不可捏造事實

格式：
THOUGHT:
<你的推理>

---

### Step 5 — ITERATE（迭代）
- 若答案仍不夠完善：
  - 使用 SolutionTool 深化分析，或
  - 使用 CriticTool 進行批判檢驗
- 持續循環直到：
  - 分析完整
  - 假設與風險已被識別
  - 結果具備決策價值

---

## 🛑 停止條件

在以下情況必須停止迭代：

- 解答已具備：
  - 結構完整
  - 有證據基礎
  - 已經過批判性檢驗
- 主要不確定性已被說明
- 使用工具已無顯著改善空間

---

## 🧾 最終輸出

當完成後，輸出：

FINAL ANSWER:
- 清晰且結構化的回答
- 必須包含：
  - 核心洞察
  - 風險與不確定性
  - 可執行建議

⚠️ 不可包含 PLAN / ACTION / OBSERVATION / THOUGHT

---

## ⚠️ 使用規則

- 每次行動前先進行 PLAN
- 不可對「原始使用者問題」直接使用 CriticTool
- CriticTool 僅可批判，不能新增分析內容
- 優先透過多輪迭代提升品質，而非一次性回答
- 必須避免無限循環，應主動收斂

---

## 🧩 風格要求

- 分析嚴謹、結構清晰
- 避免冗長與空泛敘述
- 以決策支援為導向
""")

In [ ]:
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=AGENT_SYSTEM_PROMPT,
    name = "analysis_agent"
)



In [ ]:
for update in agent.stream(
    {"messages": {"role": "user", "content": content}},
    stream_mode="updates",
):
    print(update)

In [ ]:
print(update['model']['messages'][0].content)

In [ ]:
for update in agent.stream(
    {"messages": {"role": "user", "content": content}},
    stream_mode="updates",
):
    print(update)

In [ ]:
print(update['model']['messages'][0].content)

)## 如何讓Tool接收複數的變數?

本節說明如何建立多變量輸入工具（如 SearchTool)

In [ ]:
from openai import OpenAI

client = OpenAI()

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


class Inputs(BaseModel):
    query: str = Field(description="User query")
    country_code: str = Field(description="ISO 3166-1 alpha-2 suggested by the language of the user query")

In [ ]:
from typing import Annotated

class SearchTool(BaseTool):

    input_output_parser: PydanticOutputParser = PydanticOutputParser(pydantic_object=Inputs)
    input_format_instructions: str = input_output_parser.get_format_instructions()
    
    name:str = "Websearch-tool"
    description_template:str = dedent("""\    
    Use this tool to collect information from the internet, when you are not sure you know the answer.
    The input contains the user's question `query` and the ISO 3166-1 alpha-2 `country_code` inferred from the user's language.
    input format instructions: {input_format_instructions}
    """)

    description: str = description_template.format(input_format_instructions=input_format_instructions)
    
    def _run(self, **input):
        
        # 檢查 input 是否是 有多個輸入
        print(input)

        # 有時候會有兩種輸入結構:
        """
        input = {'input': {'query': query, 'country_code': country_code}}

        或是

        input = {'query': query, 'country_code': country_code}
        """

        input = input.get('input', input)
        
        query = input["query"]
        country_code = input["country_code"]
        
        messages = [{"role": "user",
                     "content": query}]

        response = client.responses.create(
                    model="gpt-4o-mini",
                    tools=[
                        {"type": "web_search",
                         "user_location":{
                             "type": "approximate",
                             "country": country_code,
                         },
                        "search_context_size": "medium"
                        }],
                    tool_choice="auto",
                    input=messages)
        
        return response.output_text
    
    def _arun(self, query: str):
        raise NotImplementedError("This tool does not support async")

In [ ]:
(發生在大語言模型的知識截止時間以後)AGENT_WEB_SYSTEM_PROMPT = dedent("""\
你是一個進階的研究型 ReAct Agent，專門透過「多輪搜尋 + 批判性推理」產出高品質、具證據基礎的答案。

你可以使用以下工具：
- Websearch-tool：用於從網路取得最新(發生在大語言模型的知識截止時間以後)或不確定的資訊

---

## 🎯 任務目標

1. 完整理解使用者問題
2. 在行動前先制定清晰搜尋與分析策略
3. 透過多輪搜尋逐步補齊資訊
4. 對搜尋結果進行交叉驗證與批判性評估
5. 辨識資訊缺口並持續補強
6. 最終整合為完整、可靠、可用於決策的答案

---

## 🧠 推理流程（ReAct + 搜尋迭代）

你必須嚴格遵循以下循環流程：

---

### Step 1 — PLAN（規劃）
- 拆解問題
- 識別：
  - 核心問題
  - 需要查證的資訊
  - 潛在關鍵字（search queries）
- 制定搜尋策略（可多步）

格式：
PLAN:
- Step 1: ...
- Step 2: ...

---

### Step 2 — ACT（執行搜尋）
當你需要資訊時，使用 Websearch-tool

格式：
ACTION:
Tool: Websearch-tool
Input: {
  "query": "<具體且優化後的搜尋問題>",
  "country_code": "<ISO 3166-1 alpha-2 code>"
}

規則：
- query 必須具體（避免模糊問題）
- 必要時逐步細化查詢
- country_code 依語言推斷：
  - 繁體中文 → TW
  - 英文 → US
  - 日文 → JP

---

### Step 3 — OBSERVATION（觀察）
- 記錄搜尋結果
- 不可修改或幻想內容

格式：
OBSERVATION:
<搜尋結果>

---

### Step 4 — THOUGHT（推理 / 評估）
你必須評估：

- 資訊是否足夠？
- 是否存在矛盾或不一致？
- 是否需要更多來源驗證？
- 是否需要拆分問題再搜尋？

規則：
- 必須逐步推理（但保持精簡）
- 不可捏造資訊
- 必須指出不確定性

格式：
THOUGHT:
<你的推理>

---

### Step 5 — ITERATE（多輪搜尋）
若存在以下情況，必須繼續搜尋：

- 資訊不完整
- 缺乏證據
- 出現矛盾
- 問題仍過於抽象

策略：
- 改寫 query（更精準）
- 拆成子問題
- 搜尋不同角度（例如：數據 / 專家觀點 / 最新發展）

持續循環：
PLAN → ACTION → OBSERVATION → THOUGHT

---

## 🛑 停止條件

在以下情況必須停止搜尋：

- 已有足夠資訊支持完整回答
- 關鍵不確定性已被說明
- 再搜尋無明顯價值提升

---

## 🧾 最終輸出

當完成後，輸出：

FINAL ANSWER:
- 結構清晰、邏輯完整
- 必須包含：
  - 核心結論
  - 關鍵證據或觀察
  - 不確定性與限制
  - （如適用）實務建議

⚠️ 不可包含 PLAN / ACTION / OBSERVATION / THOUGHT

---

## ⚠️ 嚴格規則

- 每次行動前必須有 PLAN
- 不可跳過 THOUGHT
- 不可憑空回答（不確定就搜尋）
- 不可只搜尋一次就結束（除非已充分）
- 必須進行多來源驗證（若重要）
- Action Input 必須是合法 JSON
- 不可偽造搜尋結果

---

## 🧩 搜尋最佳實務（重要）

你應該：

- 將模糊問題轉成具體搜尋語句
- 使用不同角度查詢：
  - 定義
  - 數據
  - 趨勢
  - 專家觀點
- 避免過長 query
- 優先拆解問題再查

錯誤示例 ❌：
"AI未來如何？"

正確示例 ✅：
"2025 AI trends industry report key developments"
"生成式AI市場規模 2025 預測"

---

## 🧠 思考風格

- 嚴謹但精簡
- 以證據為導向
- 持續質疑資訊完整性
- 主動發現資訊缺口

---

## 🎯 目標

你不是在「回答問題」，而是在：

👉 建立一個「經過搜尋驗證的可靠結論」
""")

In [ ]:
tools = [SearchTool()]

agent_websearch = create_agent(name='websearch agent',
                               model=model,
                               tools=tools,
                               system_prompt=AGENT_WEB_SYSTEM_PROMPT)

In [ ]:
from langchain_core.messages import HumanMessage

for update in agent_websearch.stream(
    {"messages": {"role": "user", "content": "台灣剴剴案最新進度 (2026年)"}},
    stream_mode="updates",
):
    print(update)

In [ ]:
print(update['model']['messages'][0].content)

## Agent 範例二

數據分析

In [ ]:
os.listdir("tutorial")

In [ ]:
import json

import pandas as pd

filename = os.path.join("tutorial", "numerical_analysis", "2024學年度_10411-03-01-2_臺中市高級中等學校外國學生數.json")

with open(filename, 'r', encoding='utf-8') as f:
    data = json.load(f)

In [ ]:
df = pd.DataFrame(data=data)

filename = os.path.join("tutorial", "week_7", "測試.csv")

df.to_csv(filename)

要處理數據的第一步是了解數據的格式和內容

我們使用代碼執行的方式，來取得數據的格式和內容

In [ ]:
import io
import os
from contextlib import redirect_stdout
from pydantic import BaseModel, Field
from textwrap import dedent

from langchain.tools import BaseTool, ToolRuntime
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables import Runnable


class Context(BaseModel):
    directory: str
    schema_output: str = ""
    pandas_output: str = ""


SCHEMA_SYSTEM_PROMPT = dedent("""
# Role
你是一位專業的 Python 資料分析師，精通 pandas 資料處理。你的任務是為指定的 CSV 檔案生成精確、可執行的 Python 代碼。

# Goal
生成一段可直接執行的 Python 代碼，使用 pandas 讀取 CSV 檔案，並輸出該檔案的結構摘要與前五筆資料。

# Input
- <file>: 待分析的 CSV 檔案完整路徑

# Rule
- 使用 `pd.read_csv()` 讀取 CSV 檔案，指定 `encoding='utf-8'`
- 使用 `df.info()` 輸出人類可讀的完整摘要（欄位名、非空值數量、資料型別）
- 使用 `print(df.head(5).to_string())` 輸出前五筆資料
- 輸出的代碼必須是純 Python 代碼，不含任何 markdown 標記或解釋文字

# Constraints
- 嚴禁輸出 markdown 代碼塊標記（如 ```python 或 ```）
- 嚴禁在代碼前後添加任何說明、註解或對話文字
- 嚴禁使用任何未安裝的第三方套件（僅限 pandas 與 Python 標準庫）
- 嚴禁修改、刪除或寫入任何檔案
- 嚴禁使用網路請求或外部 API

# Reasoning (Chain of Thought)
請依以下步驟逐步推理，每完成一步再進行下一步：

Step 1: [狀態確認] 確認輸入的 CSV 檔案路徑 {file}，判斷是否需要特殊編碼處理
Step 2: [關鍵分析] 確定讀取策略：使用 pd.read_csv()，設定適當的 encoding 參數
Step 3: [推理展開] 構建輸出邏輯：先呼叫 df.info() 輸出結構摘要，再呼叫 print(df.head(5).to_string()) 輸出前五筆
Step 4: [驗證檢查] 檢查代碼是否僅包含必要的 import 與執行語句，無多餘內容
Step 5: [整合輸出] 輸出純 Python 代碼字串，不含任何格式包裝
""")


class SchemaTool(BaseTool):
    name: str = "schema_tool"
    description: str = dedent("""
    Reads all files in a given directory and returns the schema (structure summary) and first 5 rows of each file. Uses pandas to analyze CSV files, providing column names, data types, non-null counts, and sample data. Use this tool when you need to understand the structure and content of data files before further processing.
    """)

    pipeline: Runnable

    @classmethod
    def create(cls, llm: Runnable):

        input_ = {
            "system": {"template": SCHEMA_SYSTEM_PROMPT},
            "human": {
                "template": dedent("""
                    <file>: {file}
                """),
                "input_variable": ["file"]
            }
        }
        pipeline = build_standard_chat_prompt_template(input_) | llm | StrOutputParser()

        return cls(pipeline=pipeline)

    def _run(self, runtime: ToolRuntime[Context], **input):
        directory = runtime.context.directory
    
        raw_output = ""
        
        for f in os.listdir(directory):
            
            if not f.endswith(".csv"):
                continue
                
            file = os.path.join(directory, f)
            
            code = self.pipeline.invoke({
                "file": file
            })
    
            stdout_capture = io.StringIO()
            try:
                with redirect_stdout(stdout_capture):
                    exec(code, {"__builtins__": __builtins__})
                output = stdout_capture.getvalue()
            except Exception as e:
                output = f"EXECUTION ERROR: {str(e)}"

            raw_output += f"-{file}: {output}\n\n"

        runtime.context.schema_output = raw_output
            
        return raw_output

    async def _arun(self, runtime: ToolRuntime[Context]):

        return "Not implemented Yet"

In [ ]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain.agents.middleware.tool_retry import ToolRetryMiddleware
from langchain.agents.middleware.tool_call_limit import ToolCallLimitMiddleware
from langchain_core.messages import HumanMessage


AGENT_INSTRUCTION = dedent("""
# Role
你是一位專業的數據分析顧問，擅長解讀資料結構並提供清晰的分析建議。你的工作方式是先了解資料的樣貌，再根據使用者需求給出精準的回應。

# Goal
協助使用者理解資料內容並回答數據分析相關問題。

# Tool
你可以使用以下工具來完成任務：

- **schema_tool**: 讀取指定目錄中的所有 CSV 檔案，返回每個檔案的結構摘要（欄位名稱、資料型別、非空值數量）與前五筆範例資料。適用於需要了解資料結構與內容的場景。

# Input
使用者會以自然語言描述他們的數據分析需求或問題。

# Rule
- 當使用者詢問關於資料檔案的內容、結構或分析建議時，先調用工具獲取資料資訊
- 根據工具返回的結構摘要與前五筆資料，解讀每個欄位的意義與資料型別
- 針對使用者的具體問題，基於實際的資料結構給出分析建議或答案
- 若工具返回錯誤，如實告知使用者並建議檢查檔案格式

# Constraints
- 嚴禁在未調用工具的情況下憑空猜測資料內容
- 嚴禁對資料進行任何寫入、修改或刪除操作
- 回答必須基於工具返回的實際資料，不得虛構

# Reasoning (ReAct)
使用 ReAct（Reasoning + Acting）框架，在「推理」與「行動」之間交替進行：

Thought: 分析使用者需求，判斷是否需要調用工具，以及需要哪些資訊
Action: 選擇並調用合適的工具，傳入必要參數
Observation: 仔細觀察工具返回的結果，提取關鍵資訊
Thought: 根據觀察結果進行分析推理，判斷是否需要進一步行動
...（重複 Thought → Action → Observation 循環，直到足以回答問題）
Final Answer: 給出清晰、有條理的最終回答
""")


llm = ChatOllama(model='deepseek-v4-pro:cloud',
                 base_url='https://ollama.com',
                 name='main', temperature=0)


tools = [SchemaTool.create(llm=llm)]

agent = create_agent(
    model=llm,
    name="analysis_agent",
    tools=tools,
    system_prompt=AGENT_INSTRUCTION,
    middleware=[
        ToolRetryMiddleware(max_retries=2),
        ToolCallLimitMiddleware(exit_behavior='end',
                                run_limit=1,
                                tool_name="schema_tool")
    ]
)

`ToolRetryMiddleware` 是一個中介軟體，用來自動處理工具執行失敗的情況。  

**功能說明：**

- 當工具執行過程中發生錯誤（raise exception）時  
- 中介軟體會自動等待（wait）一段時間後再重新嘗試執行（retry）  
- 這樣可以提高代理程式的穩定性，尤其是在工具偶爾失敗或外部服務暫時不可用時  

使用 `ToolRetryMiddleware` 可以讓你的工具更具容錯能力，避免一次錯誤導致整個流程中斷。

`ToolCallLimitMiddleware` 設置工具調用的次數上限，避免Agent進入無限循環

你可以看到當指定的Tool的調動次數到達上限後，直接跳出。

exit_behavior：當超出限制時的處理方式

- 'continue'：對超出限制的工具回傳錯誤訊息並阻擋其執行，其餘工具可繼續運作；由模型自行決定何時結束
- 'error'：拋出 `ToolCallLimitExceededError` 例外
- 'end'：立即停止執行，並回傳一個 `ToolMessage` + AI 訊息（僅針對該次超出限制的單一工具呼叫）；若同時存在多個平行工具呼叫或多個待處理的工具呼叫，則會拋出 `NotImplementedError`

In [ ]:
context = Context(directory="tutorial/week_7")

`Context`

有時候你會需要一些"常數"

像是客戶ID，偏好等等的訊息，一些我們想要鎖死的資訊，要如何放入Agent系統裡，讓它們可以被工具取用?

在下面的例子中，我們將課程大綱，背景資訊作為Context傳入工具中

In [ ]:
agent_input = {"messages": HumanMessage(content="顯示所有檔案的schema")}

In [ ]:
for update in agent.stream(
    agent_input,
    context=context,
    stream_mode="updates",
):
    # 只保留 model 和 tool 兩個 key
    for key, value in update.items():
        if key == "model":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].tool_calls)
        if key == "tools":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].name)

開始加深: 加入 pandas 分析

In [ ]:
from typing import List


PANDAS_SYSTEM_PROMPT = dedent("""
# Role
你是一位專業的 Python 資料科學家，精通 pandas、scikit-learn、scipy 等資料處理與科學計算庫。你的任務是根據使用者指定的 CSV 檔案，生成精確、可執行的 Python 代碼來進行數據處理與分析。

# Goal
生成一段可直接執行的 Python 代碼，使用 pandas、scikit-learn、scipy 等專業庫對指定的 CSV 檔案進行數據處理與分析，並輸出處理結果。

# Input
- <files>: 待處理的 CSV 檔案完整路徑列表
- <context>: schema_tool 回傳的檔案結構摘要（欄位名稱、資料型別、範例資料）
- <user_query>: 使用者的原始分析需求或問題

# Rule
- 使用 `pd.read_csv()` 讀取每個 CSV 檔案，指定 `encoding='utf-8'`
- 根據 <user_query> 與 <context> 中的欄位結構，選用合適的庫進行數據處理與分析：
  - **pandas**: 資料讀取、篩選、分組、聚合、合併（`pd.merge` / `pd.concat`）、排序、描述性統計
  - **scikit-learn**: 資料預處理（標準化、編碼、降維）、模型訓練與預測、評估指標
  - **scipy**: 統計檢定（t-test、chi-square、ANOVA）、優化、信號處理、稀疏矩陣運算
- 使用 `print()` 輸出處理結果，確保輸出清晰可讀
- 輸出的代碼必須是純 Python 代碼，不含任何 markdown 標記或解釋文字

# Constraints
- 嚴禁輸出 markdown 代碼塊標記（如 ```python 或 ```）
- 嚴禁在代碼前後添加任何說明、註解或對話文字
- 僅限使用 pandas、scikit-learn、scipy、numpy 與 Python 標準庫，嚴禁使用其他未安裝的第三方套件
- 嚴禁修改、刪除或寫入任何檔案
- 嚴禁使用網路請求或外部 API

# Reasoning (Chain of Thought)
請依以下步驟逐步推理，每完成一步再進行下一步：

Step 1: [狀態確認] 確認輸入的 CSV 檔案列表 {files}，判斷檔案數量、結構與可能的關聯性
Step 2: [需求分析] 根據 <user_query> 與 <context> 中的欄位資訊，選擇最合適的庫與方法（pandas 處理 / sklearn 建模 / scipy 檢定）
Step 3: [推理展開] 構建處理流程：讀取 → 預處理 → 分析/建模 → 輸出結果
Step 4: [驗證檢查] 檢查代碼是否僅包含必要的 import 與執行語句，無多餘內容
Step 5: [整合輸出] 輸出純 Python 代碼字串，不含任何格式包裝
""")


class FilesInputs(BaseModel):
    files: List[str] = Field(
        description="List of CSV file names to process. Provide the filenames (without directory path) of the data files you want to analyze or transform.")
    user_query: str = Field(
        description="The user's original question or analysis requirement in natural language.")


class PandasTool(BaseTool):
    name: str = "pandas_tool"

    description_template: str = dedent("""
Processes specified CSV files using Python and pandas. Provide a list of file names to perform data operations such as filtering, grouping, aggregation, merging, sorting, and statistical analysis. Use this tool when you need to transform, analyze, or compute results from data files.

{input_format_instructions}
    """)

    input_parser: PydanticOutputParser = PydanticOutputParser(pydantic_object=FilesInputs)
    input_format_instructions: str = input_parser.get_format_instructions()

    description: str = description_template.format(input_format_instructions=input_format_instructions)

    pipeline: Runnable

    @classmethod
    def create(cls, llm: Runnable):

        input_ = {
            "system": {"template": PANDAS_SYSTEM_PROMPT},
            "human": {
                "template": dedent("""
                    <files>: {files}
                    <context>: {context}
                    <user_query>: {user_query}
                """),
                "input_variables": ["files", "context", "user_query"]
            }
        }
        pipeline = build_standard_chat_prompt_template(input_) | llm | StrOutputParser()

        return cls(pipeline=pipeline)

    def _run(self, runtime: ToolRuntime[Context], **input):
        directory = runtime.context.directory

        args = input.get("input", input)

        files = args['files']
        user_query = args['user_query']
        
        # Prepend cached schema output to ensure full schema info is available
        if runtime.context.schema_output:
            context = f"Schema Information:\n{runtime.context.schema_output}\n\n"

        code = self.pipeline.invoke({
            "files": [os.path.join(directory, f) for f in files],
            "context": context,
            "user_query": user_query
        })

        stdout_capture = io.StringIO()
        try:
            with redirect_stdout(stdout_capture):
                exec(code, {"__builtins__": __builtins__})
            output = stdout_capture.getvalue()
        except Exception as e:
            output = f"EXECUTION ERROR: {str(e)}"

        return output

    async def _arun(self, runtime: ToolRuntime[Context]):

        return "Not implemented Yet"

In [ ]:
AGENT_INSTRUCTION_VERSION_2 = dedent("""
# Role
你是一位專業的數據分析顧問，擅長解讀資料結構並提供清晰的分析建議。你的工作方式是先了解資料的樣貌，再根據使用者需求給出精準的回應。

# Goal
協助使用者理解資料內容並回答數據分析相關問題。

# Tool
你可以使用以下工具來完成任務：

- **schema_tool**: 讀取指定目錄中的所有 CSV 檔案，返回每個檔案的結構摘要（欄位名稱、資料型別、非空值數量）與前五筆範例資料。注意：此工具僅提供結構資訊與極少量範例，無法用於統計、篩選或計算。
- **pandas_tool**: 使用 Python 與 pandas、scikit-learn、scipy 等專業庫對指定的 CSV 檔案進行數據處理與分析。支援篩選、分組、聚合、合併、統計檢定、機器學習建模等操作。這是唯一能對資料進行實際計算與分析的工具。

# Input
使用者會以自然語言描述他們的數據分析需求或問題。

# Rule
- 第一步：調用 `schema_tool` 了解有哪些檔案、每個檔案的欄位結構與資料型別
- 第二步：根據 schema_tool 返回的欄位資訊，調用 `pandas_tool` 對相關檔案進行實際的數據處理與分析
- 重要：調用 pandas_tool 時，必須將 schema_tool 的完整輸出作為 `context` 參數傳入，並將使用者的原始問題作為 `user_query` 參數傳入
- 重要：schema_tool 僅提供結構與 5 筆範例，任何涉及篩選、統計、聚合、計數、排序、建模的操作都必須透過 pandas_tool 完成
- 調用 pandas_tool 時，從 schema_tool 的結果中選取相關的檔案名稱傳入 `files` 參數
- 若工具返回錯誤，如實告知使用者並建議檢查檔案格式或需求描述

# Constraints
- 嚴禁在未調用工具的情況下憑空猜測資料內容
- 嚴禁僅憑 schema_tool 的 5 筆範例資料就回答需要統計或計算的問題
- 嚴禁對資料進行任何寫入、修改或刪除操作
- 回答必須基於 pandas_tool 的實際計算結果，不得虛構
- 回答語言: 繁體中文

# Reasoning (ReAct)
使用 ReAct（Reasoning + Acting）框架，在「推理」與「行動」之間交替進行：

Thought: 分析使用者需求，判斷這是否為純結構問題（只需 schema_tool）還是數據分析問題（需要 pandas_tool 進行計算）
Action: 若需要了解結構，調用 schema_tool；若需要計算分析，調用 pandas_tool
Observation: 仔細觀察工具返回的結果，提取關鍵資訊
Thought: 若已調用 schema_tool 但問題需要數據計算，則必須繼續調用 pandas_tool，不可在此停止
...（重複 Thought → Action → Observation 循環，直到足以回答問題）
Final Answer: 基於 pandas_tool 的計算結果，給出清晰、有條理的最終回答
""")


llm = ChatOllama(model='deepseek-v4-pro:cloud',
                 base_url='https://ollama.com',
                 name='main', temperature=0)


tools = [SchemaTool.create(llm=llm),
         PandasTool.create(llm=llm)]

agent = create_agent(
    model=llm,
    name="analysis_agent_version_2",
    tools=tools,
    system_prompt=AGENT_INSTRUCTION_VERSION_2,
    middleware=[
        ToolRetryMiddleware(max_retries=2),
    ]
)

In [ ]:
agent_input = {"messages": HumanMessage(content="顯示外國學生數量")}
for update in agent.stream(
    agent_input,
    context=context,
    stream_mode="updates",
):
    # 只保留 model 和 tool 兩個 key
    for key, value in update.items():
        if key == "model":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].tool_calls)
        if key == "tools":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].name)

單一檔案測試通過，加大難度: 多檔案

In [ ]:
directory = os.path.join("tutorial", "week_7", "多檔案測試")

In [ ]:
context = Context(directory=directory)

agent = create_agent(
    model=llm,
    name="analysis_agent_version_2",
    tools=tools,
    system_prompt=AGENT_INSTRUCTION_VERSION_2,
    middleware=[
        ToolRetryMiddleware(max_retries=2),
    ]
)

In [ ]:
agent_input = {"messages": HumanMessage(content="台中市學生從2021年到2024年的變化")}

In [ ]:
for update in agent.stream(
    agent_input,
    context=context,
    stream_mode="updates",
):
    # 只保留 model 和 tool 兩個 key
    for key, value in update.items():
        if key == "model":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].tool_calls)
        if key == "tools":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].name)

有數據之後，就可以寫報告

In [ ]:
from textwrap import dedent
from pydantic import BaseModel, Field
from langchain.tools import BaseTool, ToolRuntime
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables import Runnable

REPORT_SYSTEM_PROMPT = dedent("""
# Role
你是一位專業的報告撰寫專家，擅長將數據分析結果轉化為結構清晰、易讀的書面報告。

# Goal
根據使用者需求與分析結果，生成一份完整的文字報告。報告應包含：標題、摘要、數據分析結果、結論與建議。

# Input
- <requirement>: 使用者的原始需求描述
- <analysis_result>: 數據分析工具回傳的結果

# Rule
- 報告必須使用繁體中文撰寫
- 報告結構：標題 → 摘要 → 分析內容 → 結論
- 保留分析結果中的所有關鍵數字與統計數據
- 使用清晰的段落分隔，便於後續 Word 排版
- 以純文字輸出，不使用 markdown 語法

# Constraints
- 不得虛構或竄改分析結果中的數據
- 不得添加未經分析驗證的推測
""")


class ReportInputs(BaseModel):
    requirement: str = Field(
        description="The user's original requirement or question that triggered the analysis.")
    analysis_result: str = Field(
        description="The raw output from data analysis tools (e.g., pandas_tool results) to be turned into a report.")


class ReportTool(BaseTool):
    name: str = "report_tool"

    description_template: str = dedent("""
    Generates a structured text report based on user requirements and data analysis results.
    Use this tool when you need to produce a formal written report from analysis output.

    {input_format_instructions}
    """)

    input_parser: PydanticOutputParser = PydanticOutputParser(pydantic_object=ReportInputs)
    input_format_instructions: str = input_parser.get_format_instructions()
    description: str = description_template.format(input_format_instructions=input_format_instructions)

    pipeline: Runnable

    @classmethod
    def create(cls, llm: Runnable):
        input_ = {
            "system": {"template": REPORT_SYSTEM_PROMPT},
            "human": {
                "template": dedent("""
                    <requirement>: {requirement}
                    <analysis_result>: {analysis_result}
                """),
                "input_variables": ["requirement", "analysis_result"]
            }
        }
        pipeline = build_standard_chat_prompt_template(input_) | llm | StrOutputParser()
        return cls(pipeline=pipeline)

    def _run(self, runtime: ToolRuntime, **input):
        args = input.get("input", input)
        requirement = args["requirement"]
        analysis_result = args["analysis_result"]

        report = self.pipeline.invoke({
            "requirement": requirement,
            "analysis_result": analysis_result
        })

        return report

    async def _arun(self, runtime: ToolRuntime):
        return "Not implemented yet"


將 report_tool 生成的內容轉換成word輸出

In [ ]:
import io
import os
from contextlib import redirect_stdout
from pydantic import BaseModel, Field

WORD_FORMAT_SYSTEM_PROMPT = dedent("""
# Role
你是一位專業的 Python 文檔自動化專家，擅長使用 python-docx 庫將文字報告排版為精美的 Word 文件。

# Goal
根據輸入的報告文字，生成一段完整的 Python 代碼，使用 python-docx 創建格式化的 Word 文件並儲存。

# Input
- <report>: 需要排版的報告文字內容
- <output_path>: Word 文件的輸出路徑

# Rule
- 使用 `from docx import Document` 創建文件
- 使用 `docx.shared.Pt` 設定字型大小
- 使用 `docx.shared.RGBColor` 設定顏色
- 使用 `docx.enum.text.WD_ALIGN_PARAGRAPH` 設定對齊
- 標題使用 18pt 粗體置中
- 章節標題使用 14pt 粗體
- 內文使用 12pt 正常字體
- 設定頁面邊距為 2.54cm（標準 A4）
- 最後使用 `doc.save(output_path)` 儲存
- 使用 `print()` 輸出成功訊息

# Constraints
- 請輸出純 Python 代碼，不要用 ```python 或任何 markdown 代碼塊包裝，直接輸出可執行的原始代碼即可。
- 代碼必須完整可執行，包含所有必要的 import
- 不要使用未安裝的第三方庫

# Reasoning (Chain of Thought)
Step 1: 解析報告結構，識別標題、章節、段落
Step 2: 規劃 Word 排版格式（字型、大小、對齊）
Step 3: 生成完整的 python-docx 代碼
Step 4: 確認代碼包含 save 操作
""")


class WordFormatInputs(BaseModel):
    report: str = Field(
        description="The full text report content to be formatted into a Word document.")
    output_path: str = Field(
        default="report_output.docx",
        description="File path where the generated .docx file will be saved.")


class WordFormatTool(BaseTool):
    name: str = "word_format_tool"

    description_template: str = dedent("""
    Formats a text report into a Word (.docx) document using python-docx.
    Generates and executes Python code to create a professionally formatted Word file.
    Use this tool when you need to convert a plain text report into a styled Word document.

    {input_format_instructions}
    """)

    input_parser: PydanticOutputParser = PydanticOutputParser(pydantic_object=WordFormatInputs)
    input_format_instructions: str = input_parser.get_format_instructions()
    description: str = description_template.format(input_format_instructions=input_format_instructions)

    pipeline: Runnable

    @classmethod
    def create(cls, llm: Runnable):
        input_ = {
            "system": {"template": WORD_FORMAT_SYSTEM_PROMPT},
            "human": {
                "template": dedent("""
                    <report>: {report}
                    <output_path>: {output_path}
                """),
                "input_variables": ["report", "output_path"]
            }
        }
        pipeline = build_standard_chat_prompt_template(input_) | llm | StrOutputParser()
        return cls(pipeline=pipeline)

    def _run(self, runtime: ToolRuntime, **input):

        args = input.get("input", input)
        report = args["report"]
        output_path = args.get("output_path", "report_output.docx")

        code = self.pipeline.invoke({
            "report": report,
            "output_path": output_path
        })

        # 有時候代碼會有 ```python ... ``` 的結構，我們這裡強制移除掉
        code = code.replace("```python", "").replace("```", "")
        
        stdout_capture = io.StringIO()
        try:
            with redirect_stdout(stdout_capture):
                exec(code, {"__builtins__": __builtins__})
            output = stdout_capture.getvalue()
            if not output:
                output = f"Word document saved to: {output_path}"
            return output
        except Exception as e:
            output = f"WORD FORMAT ERROR: {str(e)}"
            return f"Code:\n{code}\n\n\nError: {output}"

    async def _arun(self, runtime: ToolRuntime):
        return "Not implemented yet"


因為 word_format_tool 涉及代碼操作

所以還是有可能出錯。因此我們多放入一個工具負責校正

In [ ]:
from pydantic import BaseModel, Field

CODE_FIX_SYSTEM_PROMPT = dedent("""
# Role
你是一位專業的 Python 代碼調試專家，擅長快速定位並修復 python-docx 排版代碼中的錯誤。

# Goal
根據錯誤訊息與失敗的代碼，生成修正後的完整 Python 代碼。

# Input
- <error_message>: 執行失敗的錯誤訊息
- <failed_code>: 導致錯誤的原始代碼
- <report>: 原始報告文字（供參考）
- <output_path>: Word 輸出路徑

# Rule
- 仔細分析錯誤訊息，定位根本原因
- 修正代碼中的語法錯誤、API 使用錯誤或邏輯問題
- 保持原有的排版意圖與格式設定
- 輸出完整的可執行代碼，而非僅修正片段
- 常見問題檢查：import 缺失、API 名稱錯誤、參數型別錯誤、NoneType 存取

# Constraints
- 請輸出純 Python 代碼，不要用 ```python 或任何 markdown 代碼塊包裝，直接輸出可執行的原始代碼即可。
- 代碼必須完整可執行
- 不要引入新的第三方庫

# Reasoning (Chain of Thought)
Step 1: 閱讀錯誤訊息，定位錯誤類型和位置
Step 2: 檢查對應的代碼行，找出根本原因
Step 3: 修正錯誤，確保代碼邏輯正確
Step 4: 輸出完整的修正代碼
""")


class CodeFixInputs(BaseModel):
    error_message: str = Field(
        description="The full error message or traceback from the failed code execution.")
    failed_code: str = Field(
        description="The complete Python code that produced the error.")
    report: str = Field(
        default="",
        description="The original report text that was being formatted, for context.")
    output_path: str = Field(
        default="report_output.docx",
        description="The target output path for the Word document.")


class CodeFixTool(BaseTool):
    name: str = "code_fix_tool"

    description_template: str = dedent("""
    Fixes Python code that failed during Word document generation.
    Analyzes the error message and failed code, then produces corrected code.
    Use this tool when WordFormatTool returns an error and the code needs debugging.

    {input_format_instructions}
    """)

    input_parser: PydanticOutputParser = PydanticOutputParser(pydantic_object=CodeFixInputs)
    input_format_instructions: str = input_parser.get_format_instructions()
    description: str = description_template.format(input_format_instructions=input_format_instructions)

    pipeline: Runnable

    @classmethod
    def create(cls, llm: Runnable):
        input_ = {
            "system": {"template": CODE_FIX_SYSTEM_PROMPT},
            "human": {
                "template": dedent("""
                    <error_message>: {error_message}
                    <failed_code>: {failed_code}
                    <report>: {report}
                    <output_path>: {output_path}
                """),
                "input_variables": ["error_message", "failed_code", "report", "output_path"]
            }
        }
        pipeline = build_standard_chat_prompt_template(input_) | llm | StrOutputParser()
        return cls(pipeline=pipeline)

    def _run(self, runtime: ToolRuntime, **input):

        args = input.get("input", input)
        error_message = args["error_message"]
        failed_code = args["failed_code"]
        report = args.get("report", "")
        output_path = args.get("output_path", "report_output.docx")

        print("\nFailed Code")
        print("="*20)
        print(failed_code)
        print("="*20)
        
        fixed_code = self.pipeline.invoke({
            "error_message": error_message,
            "failed_code": failed_code,
            "report": report,
            "output_path": output_path
        })

        print("\nFixed Code")
        print("="*20)
        print(fixed_code)
        print("="*20)
        
        stdout_capture = io.StringIO()
        try:
            with redirect_stdout(stdout_capture):
                exec(fixed_code, {"__builtins__": __builtins__})
            output = stdout_capture.getvalue()
            if not output:
                output = f"Word document saved to: {output_path}"
        except Exception as e:
            output = f"WORD FORMAT ERROR: {str(e)}"

        return output

    async def _arun(self, runtime: ToolRuntime):
        return "Not implemented yet"

In [ ]:
from langchain.agents.middleware.model_call_limit import ModelCallLimitMiddleware


FULL_AGENT_INSTRUCTION = dedent("""
# Role
你是一位專業的數據分析與報告自動化顧問。你的工作流程是：分析數據 → 生成報告 → 排版 Word → 修正錯誤。

# Goal
根據使用者需求，完成從數據分析到 Word 報告輸出的完整流程。

# Tool
你可以使用以下工具：
- **schema_tool**: 讀取 CSV 檔案結構
- **pandas_tool**: 執行數據分析（需傳入 context=schema_tool 輸出、user_query=使用者原始問題）
- **report_tool**: 根據分析結果生成文字報告（需傳入 requirement=使用者需求、analysis_result=pandas_tool 輸出）
- **word_format_tool**: 將報告排版為 Word 文件（需傳入 report=report_tool 輸出）
- **code_fix_tool**: 修復 Word 排版代碼錯誤（需傳入 error_message、failed_code、report）

# Rule
1. 先使用 schema_tool 了解資料結構
2. 使用 pandas_tool 進行數據分析：將 schema_tool 的完整輸出作為 `context` 參數，使用者的原始問題作為 `user_query` 參數傳入
3. 使用 report_tool 生成文字報告：將使用者需求作為 `requirement`，pandas_tool 的輸出作為 `analysis_result` 傳入
4. 使用 word_format_tool 將報告轉為 Word 文件：將 report_tool 的輸出作為 `report` 參數傳入
5. 若 word_format_tool 返回錯誤（含 "WORD FORMAT ERROR"），使用 code_fix_tool 修正代碼後重試

# Constraints
- 必須按順序執行，不可跳過步驟
- 每個工具所需的參數必須正確傳入，不可遺漏
- 若工具返回錯誤，如實告知使用者
""")


all_tools = [
    SchemaTool.create(llm=llm),
    PandasTool.create(llm=llm),
    ReportTool.create(llm=llm),
    WordFormatTool.create(llm=llm),
    CodeFixTool.create(llm=llm),
]

full_agent = create_agent(
    model=llm,
    name="full_report_agent",
    tools=all_tools,
    system_prompt=FULL_AGENT_INSTRUCTION,
    middleware=[
        ToolRetryMiddleware(max_retries=2),
        ModelCallLimitMiddleware(run_limit=30, exit_behavior="end"),
    ]
)

print("Full report agent created with all 5 tools.")

In [ ]:
directory = os.path.join("tutorial", "week_7", "多檔案測試")

context = Context(directory=directory)

agent_input = {
    "messages": HumanMessage(content="分析台中市學生數量從2021年到2024年的變化，並生成 Word 報告")
}

In [ ]:
for update in full_agent.stream(
    agent_input,
    context=context,
    stream_mode="updates",
):
    # 只保留 model 和 tool 兩個 key
    for key, value in update.items():
        if key == "model":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].tool_calls)
        if key == "tools":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].name)


### 加入圖片

In [ ]:
class Context(BaseModel):
    directory: str
    schema_output: str = ""
    pandas_output: str = ""

In [ ]:
from typing import List


MATPLOTLIB_SYSTEM_PROMPT = dedent("""
# Role
你是一位專業的 Python 資料視覺化專家，精通 matplotlib、seaborn 等視覺化庫。你的任務是根據使用者指定的視覺化任務與數據內容，生成精確、可執行的 Python 代碼來生成圖片。

# Goal
生成一段可直接執行的 Python 代碼，使用 matplotlib 根據每個任務的描述生成對應的圖片，並將圖片儲存為指定的檔案名稱。

# Input
- <tasks>: 視覺化任務列表，每個任務包含：
  - file_name: 圖片的輸出檔案名稱（含副檔名，如 .png）
  - task: 需要完成的視覺化任務描述（例如：繪製長條圖、折線圖、圓餅圖等）
- <context>: 需要進行視覺化的數據內容（通常為 pandas_tool 或 schema_tool 回傳的數據）

# Rule
- 使用 `import matplotlib.pyplot as plt` 進行數據視覺化
- 使用 `import matplotlib` 並設定 `matplotlib.rcParams['font.sans-serif'] = ['Microsoft JhengHei']` 以支援中文字型顯示
- 根據每個 task 的 task 描述，選擇合適的圖表類型（長條圖 bar、折線圖 plot、圓餅圖 pie、散佈圖 scatter 等）
- 每個任務生成一張獨立的圖片，使用 `plt.savefig(file_name, dpi=150, bbox_inches='tight')` 儲存
- 在每個圖片生成前使用 `plt.figure()` 建立新圖表，生成後使用 `plt.close()` 關閉以避免圖表重疊
- 使用 `print(f'已儲存: {{file_name}}')` 輸出每個圖片的儲存確認訊息
- 輸出的代碼必須是純 Python 代碼，不含任何 markdown 標記或解釋文字

# Constraints
- 嚴禁輸出 markdown 代碼塊標記（如 ```python 或 ```）
- 嚴禁在代碼前後添加任何說明、註解或對話文字
- 僅限使用 matplotlib、numpy、pandas 與 Python 標準庫，嚴禁使用其他未安裝的第三方套件
- 嚴禁修改、刪除或寫入任何非圖片檔案
- 嚴禁使用網路請求或外部 API
- 嚴禁使用 `plt.show()`，僅使用 `plt.savefig()` 儲存圖片

# Reasoning (Chain of Thought)
請依以下步驟逐步推理，每完成一步再進行下一步：

Step 1: [狀態確認] 確認輸入的 tasks 列表，逐一檢視每個任務的 file_name 與 task 描述
Step 2: [數據解析] 根據 <context> 中的數據內容，解析數據結構，確認可用於視覺化的欄位與數值
Step 3: [圖表選擇] 針對每個 task，根據其 task 描述選擇最合適的圖表類型與視覺化方式
Step 4: [代碼構建] 為每個任務構建完整的 matplotlib 代碼：建立 figure → 繪製圖表 → 設定標題/標籤 → savefig → close
Step 5: [驗證檢查] 檢查代碼是否僅包含必要的 import 與執行語句，無多餘內容，且每個任務都有對應的 savefig
Step 6: [整合輸出] 輸出純 Python 代碼字串，不含任何格式包裝
""")

class Task(BaseModel):
    file_name: str = Field(description="圖片的檔案名稱")
    task: str = Field(description="需要完成的任務")


class DataInputs(BaseModel):
    tasks: List[Task] = Field(description="視覺化的任務。包含圖片檔案名稱和需要完成的任務")
    context: str = Field(description="需要進行視覺化的數據")


class MatplotlibTool(BaseTool):
    name: str = "matplotlib_tool"

    description_template: str = dedent("""
Generates data visualizations (charts, plots, graphs) using Python and matplotlib. Provide a list of visualization tasks, each with a file name and a task description, along with the data context to visualize. Use this tool when you need to create charts, bar plots, line plots, pie charts, scatter plots, or any visual representation of data.

{input_format_instructions}
    """)

    input_parser: PydanticOutputParser = PydanticOutputParser(pydantic_object=DataInputs)
    input_format_instructions: str = input_parser.get_format_instructions()

    description: str = description_template.format(input_format_instructions=input_format_instructions)

    pipeline: Runnable

    @classmethod
    def create(cls, llm: Runnable):

        input_ = {
            "system": {"template": MATPLOTLIB_SYSTEM_PROMPT},
            "human": {
                "template": dedent("""
                    <tasks>: {tasks}
                    <context>: {context}
                """),
                "input_variables": ["tasks", "context"]
            }
        }
        pipeline = build_standard_chat_prompt_template(input_) | llm | StrOutputParser()

        return cls(pipeline=pipeline)

    def _run(self, runtime: ToolRuntime[Context], **input):

        args = input.get("input", input)

        tasks = args['tasks']
        context = args['context']

        # Prepend cached outputs to ensure full context is available
        parts = []
        if runtime.context.schema_output:
            parts.append(f"Schema Information:\n{runtime.context.schema_output}")
        # if runtime.context.pandas_output:
        #     parts.append(f"Analysis Results:\n{runtime.context.pandas_output}")
        if parts:
            context = "\n\n".join(parts) + f"\n\nVisualization Request:\n{context}"

        code = self.pipeline.invoke({
            "tasks": tasks,
            "context": context,
        })

        stdout_capture = io.StringIO()
        try:
            with redirect_stdout(stdout_capture):
                exec(code, {"__builtins__": __builtins__})
            output = stdout_capture.getvalue()
        except Exception as e:
            output = f"EXECUTION ERROR: {str(e)}"

        return tasks

    async def _arun(self, runtime: ToolRuntime[Context]):

        return "Not implemented Yet"

將圖和文字進行排版

In [ ]:
from typing import List


TYPESETTING_SYSTEM_PROMPT = dedent("""
# Role
你是一位專業的報告排版專家，擅長將文字內容與圖片整合成結構清晰、美觀易讀的報告。你的任務是根據提供的文字內容與圖片資訊，生成一份排版精美的報告。

# Goal
生成一份完整的報告，將 <text> 中的文字內容與 <images> 中的圖片合理編排，使報告結構清晰、邏輯流暢、視覺美觀。

# Input
- <images>: 圖片列表，每個圖片包含：
  - file_name: 圖片的檔案名稱
  - caption: 圖片的內容描述
- <text>: 報告的文字內容

# Rule
- 根據文字內容的結構（標題、章節、段落），合理安排圖片的插入位置
- 每張圖片應搭配其 caption 作為圖說，放置在圖片下方
- 報告應包含清晰的標題、章節標題與段落分隔
- 使用 Markdown 格式輸出報告，確保排版整齊
- 圖片以 `![caption](file_name)` 的 Markdown 語法嵌入
- 根據文字內容的邏輯順序，將相關圖片插入到對應的段落附近

# Constraints
- 嚴禁修改或虛構文字內容，必須忠實呈現原始 <text>
- 嚴禁修改圖片的 file_name 或 caption
- 嚴禁添加不存在於輸入中的圖片
- 輸出的報告必須是完整的 Markdown 格式
- 嚴禁在報告前後添加任何說明、註解或對話文字

# Reasoning (Chain of Thought)
請依以下步驟逐步推理，每完成一步再進行下一步：

Step 1: [內容分析] 閱讀 <text> 的完整內容，識別標題、章節結構與關鍵段落
Step 2: [圖片匹配] 檢視 <images> 中每張圖片的 caption，判斷每張圖片與文字中哪個段落最相關
Step 3: [排版規劃] 決定每張圖片的最佳插入位置，確保圖片與相關文字內容緊密結合
Step 4: [報告構建] 以 Markdown 格式構建完整報告：標題 → 章節 → 段落（穿插圖片）→ 圖說
Step 5: [驗證檢查] 確認所有圖片都已嵌入、所有文字內容都已保留、排版結構清晰
Step 6: [整合輸出] 輸出完整的 Markdown 報告，不含任何格式包裝
""")

class Image(BaseModel):
    file_name: str = Field(description="圖片的檔案名稱")
    caption: str = Field(description="圖片的描述，用來判斷圖片應配合文字的哪個部分，以決定放置位置")


class ContentInputs(BaseModel):
    images: List[Image] = Field(description="需要排版的圖片列表，包含每張圖片的檔案名稱與內容描述")
    text: str = Field(description="報告的文字內容")


class TypesettingTool(BaseTool):
    name: str = "typesetting_tool"
    # Agent的程序會停在這裡，確保最後的輸出是排版好的內容。
    return_direct: bool = True
    description_template: str = dedent("""
Generates a well-formatted report by combining text content with images. Provide a list of images (each with a file name and caption) and the report text. The tool will intelligently arrange images within the text to create a structured, readable report in Markdown format. Use this tool when you need to create a report that integrates both textual content and visual elements.

{input_format_instructions}
    """)

    input_parser: PydanticOutputParser = PydanticOutputParser(pydantic_object=ContentInputs)
    input_format_instructions: str = input_parser.get_format_instructions()

    description: str = description_template.format(input_format_instructions=input_format_instructions)

    pipeline: Runnable

    @classmethod
    def create(cls, llm: Runnable):

        input_ = {
            "system": {"template": TYPESETTING_SYSTEM_PROMPT},
            "human": {
                "template": dedent("""
                    <images>: {images}
                    <text>: {text}
                """),
                "input_variables": ["images", "text"]
            }
        }
        pipeline = build_standard_chat_prompt_template(input_) | llm | StrOutputParser()

        return cls(pipeline=pipeline)

    def _run(self, runtime: ToolRuntime[Context], **input):
        args = input.get("input", input)

        images = args['images']
        text = args['text']

        # Prepend cached outputs to ensure full context is available
        parts = []
        if runtime.context.schema_output:
            parts.append(f"Schema Information:\n{runtime.context.schema_output}")
        if runtime.context.pandas_output:
            parts.append(f"Analysis Results:\n{runtime.context.pandas_output}")
        if parts:
            text = "\n\n".join(parts) + f"\n\nReport Text:\n{text}"

        output = self.pipeline.invoke({
            "text": text,
            "images": images,
        })

        return output

    async def _arun(self, runtime: ToolRuntime[Context]):

        return "Not implemented Yet"

建立 Agent

In [ ]:
AGENT_INSTRUCTION = dedent("""
# Role
你是一位專業的數據分析顧問，擅長解讀資料結構並提供清晰的分析建議。你的工作方式是先了解資料的樣貌，再根據使用者需求給出精準的回應。

# Goal
協助使用者理解資料內容並回答數據分析相關問題，必要時生成視覺化圖表來輔助說明。

# Tool
你可以使用以下工具來完成任務：

- **schema_tool**: 讀取指定目錄中的所有 CSV 檔案，返回每個檔案的結構摘要（欄位名稱、資料型別、非空值數量）與前五筆範例資料。注意：此工具僅提供結構資訊與極少量範例，無法用於統計、篩選或計算。
- **pandas_tool**: 使用 Python 與 pandas、scikit-learn、scipy 等專業庫對指定的 CSV 檔案進行數據處理與分析。支援篩選、分組、聚合、合併、統計檢定、機器學習建模等操作。這是唯一能對資料進行實際計算與分析的工具。
- **matplotlib_tool**: 使用 Python 與 matplotlib 將數據分析結果轉化為視覺化圖表（如長條圖、折線圖、圓餅圖、散佈圖等）。接受多個視覺化任務，每個任務包含圖片檔案名稱與任務描述，並根據提供的數據上下文生成對應圖片。注意：此工具僅用於視覺化，不能進行數據計算或分析。
- **typesetting_tool**: 使用 Python 與 LLM 將文字內容與圖片整合成結構清晰、美觀易讀的報告。接受圖片列表（含檔案名稱與描述）與文字內容，生成排版精美的 Markdown 報告。注意：此工具僅用於排版與報告生成，不能進行數據計算或分析。

# Input
使用者會以自然語言描述他們的數據分析需求或問題。

# Planning
根據使用者需求與可用工具，動態制定並更新執行計畫：

1. **初始規劃**：分析使用者需求，判斷需要哪些工具（schema_tool 了解結構、pandas_tool 計算分析、matplotlib_tool 視覺化、typesetting_tool 排版報告），制定完整的執行步驟計畫
2. **逐步執行**：按照計畫依序調用工具，每次只執行一個 Action
3. **動態調整**：每次 Action 完成後，根據 Observation 結果重新評估，更新後續計畫（新增、修改或刪除步驟）
4. **持續循環**：重複「制定計畫 → 執行 → 觀察 → 更新計畫」直到足以完整回答使用者問題

## 工具使用要點
- schema_tool 僅提供結構與 5 筆範例，無法用於統計或計算
- pandas_tool 是唯一能進行數據計算的工具，調用時完整的 schema 資訊會自動傳入，但仍需提供清晰的分析需求描述
- matplotlib_tool 僅用於視覺化，完整的 schema 與 pandas 分析結果會自動傳入，但仍需描述每張圖的任務
- typesetting_tool 僅用於排版報告，完整的 schema 與 pandas 分析結果會自動傳入，但仍需提供報告文字內容
- 若工具返回錯誤，如實告知使用者並建議檢查檔案格式或需求描述

# Constraints
- 嚴禁在未調用工具的情況下憑空猜測資料內容
- 嚴禁僅憑 schema_tool 的 5 筆範例資料就回答需要統計或計算的問題
- 嚴禁對資料進行任何寫入、修改或刪除操作
- 回答必須基於 pandas_tool 的實際計算結果，不得虛構
- 若使用者要求圖表，必須調用 matplotlib_tool 生成，不得以文字描述代替圖表
- 若使用者要求報告，必須調用 typesetting_tool 生成，不得以純文字代替排版報告

# Reasoning (ReAct)
使用 ReAct（Reasoning + Acting）框架，在「推理」與「行動」之間交替進行：

Thought: 分析使用者需求，判斷這是純結構問題（只需 schema_tool）、數據分析問題（需要 pandas_tool 進行計算）、視覺化問題（需要 matplotlib_tool 生成圖表），還是排版問題（需要 typesetting_tool 生成報告）
Action: 若需要了解結構，調用 schema_tool；若需要計算分析，調用 pandas_tool；若需要視覺化，調用 matplotlib_tool；若需要排版報告，調用 typesetting_tool
Observation: 仔細觀察工具返回的結果，提取關鍵資訊
Thought: 若已調用 schema_tool 但問題需要數據計算，則必須繼續調用 pandas_tool，不可在此停止
Thought: 若已調用 pandas_tool 但使用者要求圖表，則必須繼續調用 matplotlib_tool，將計算結果轉化為視覺化圖片
Thought: 若已調用 matplotlib_tool 且使用者要求報告，則必須繼續調用 typesetting_tool，將文字與圖片整合為排版報告
...（重複 Thought → Action → Observation 循環，直到足以回答問題）
Final Answer: 基於工具的實際結果，給出清晰、有條理的最終回答，若有生成圖片則一並附上圖片檔案名稱，若有生成報告則附上報告內容
""")


llm = ChatOllama(model='deepseek-v4-pro:cloud',
                 base_url='https://ollama.com',
                 name='main', temperature=0)

tools = [SchemaTool.create(llm=llm),
         PandasTool.create(llm=llm),
         MatplotlibTool.create(llm=llm),
         TypesettingTool.create(llm=llm)]

agent = create_agent(
    model=llm,
    name="analysis_agent",
    tools=tools,
    system_prompt=AGENT_INSTRUCTION,
    middleware=[
        ToolRetryMiddleware(max_retries=2),
    ]
)

In [ ]:
agent_input = {"messages": HumanMessage(content="將台中市學生從2021年到2024年的變化，做成一份報告")}

In [ ]:
for update in agent.stream(
    agent_input,
    context=context,
    stream_mode="updates",
):
    # 只保留 model 和 tool 兩個 key
    for key, value in update.items():
        if key == "model":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].tool_calls)
        if key == "tools":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].name)


In [ ]:
with open("report_output.txt", "w", encoding="utf-8") as f:
      f.write(value['messages'][0].content)

## 工具範本

自定義的ToolTemplate 是一個「工具範本 (Tool Template)」

主要目的是讓我們能快速建立具有統一結構的自訂工具（Tool）


In [ ]:
# 
# 主要目的是讓我們能快速建立具有統一結構的自訂工具（Tool）
# 
# 例如：查詢網路、執行程式碼、進行數學運算、檢索資料庫 等。

class ToolTemplate(BaseTool):
    # ----------- 屬性區 (Attributes) -----------
    runnable: Runnable                    # 工具實際執行邏輯 (例如某個 chain、function)
    name: str                             # 工具名稱（Agent 會用這個名稱呼叫它）
    input_parser: PydanticOutputParser    # 用於解析輸入的資料模型
    description: str                      # 工具說明文字（會顯示在 Agent 的可用工具清單中）

    @classmethod
    def create(cls, runnable: Runnable, name: str, description: str,
               input_parser: PydanticOutputParser):

        """
        這個類別方法用於「快速建立」一個 Tool 實例。
        它會自動插入描述文字 (description) 與輸入格式說明，
        讓 Agent 在使用時知道該怎麼傳入參數。
        """

        # 取得輸入格式說明（LangChain 的 Pydantic Parser 會產生格式提示）
        input_format_instruction = input_parser.get_format_instructions()

        # 將輸入格式說明嵌入到工具說明文字中
        description = description_template.format(
            input_format_instruction=input_format_instruction
        )

        # 回傳完整的 ToolTemplate 實例
        return cls(runnable=runnable, name=name, description=description,
                   input_parser=input_parser)
    
    def _run(self, **input):

        """
        工具在被 Agent 呼叫時，會執行這個方法。。
        """

        return self.runnable.invoke(input)

# ----------- 非同步版本 (尚未支援) -----------
    def _arun(self, query: str):
        raise NotImplementedError("This tool does not support async")

## Middleware - PII detection

偵測並處理對話中的個人可識別資訊（PII），可以依照不同需求使用可調整的策略。PII 偵測在以下情況中特別有用：
>- 需要符合法規要求的醫療與金融相關應用。
>- 需要清理記錄內容的客服系統。
>- 任何會處理使用者敏感資料的應用程式。

為了更好的看出這個東西的用處，我們開啟 Mlflow

### 建立MLflow監控

mlflow server --host 127.0.0.1 --port 8080

In [ ]:
from langchain.agents.middleware import PIIMiddleware

experiment = "Week-7-Guardrails"
uri = "http://127.0.0.1:8080"

mlflow.set_tracking_uri(uri=uri)

# Start or get an MLflow run explicitly
mlflow.set_experiment(experiment)

mlflow.langchain.autolog()

將`Redact`放在不同的位置，看差別在哪裡

In [ ]:
model_gpt_oss = ChatOllama(model='gpt-oss:120b-cloud',
                           base_url='https://ollama.com',
                           name='pii_tester')

In [ ]:
agent = create_agent(
    model=model_gpt_oss,
    middleware=[
        # 在將使用者輸入傳送給模型之前，先將電子郵件地址進行遮蔽（去識別化處理）
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        )]
)

In [ ]:
agent_input = {"messages": [{"role": "user", "content": "Give a quick reply to the following email address: abc@gmail.com. Include the email address in your reply"}]}

for update in agent.stream(
    agent_input,
    stream_mode="updates",
):
    # 只保留 model 和 tool 兩個 key
    for key, value in update.items():
        if key == "model":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].tool_calls)
        if key == "tools":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].name)


In [ ]:
agent = create_agent(
    model=model_gpt_oss,
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=False,
            apply_to_output=True
        )]
)

for update in agent.stream(
    agent_input,
    stream_mode="updates",
):
    # 只保留 model 和 tool 兩個 key
    for key, value in update.items():
        if key == "model":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].tool_calls)
        if key == "tools":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].name)


將strategy 從 `redact` 到 `mask`

In [ ]:
agent = create_agent(
    model=model_gpt_oss,
    middleware=[
        PIIMiddleware(
            "email",
            strategy="mask",
            apply_to_input=True,
        )]
)

for update in agent.stream(
    agent_input,
    stream_mode="updates",
):
    # 只保留 model 和 tool 兩個 key
    for key, value in update.items():
        if key == "model":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].tool_calls)
        if key == "tools":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].name)

使用strategy `hash`

In [ ]:
agent = create_agent(
    model=model_gpt_oss,
    middleware=[
        PIIMiddleware(
            "email",
            strategy="hash",
            apply_to_input=True,
        )]
)

for update in agent.stream(
    agent_input,
    stream_mode="updates",
):
    # 只保留 model 和 tool 兩個 key
    for key, value in update.items():
        if key == "model":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].tool_calls)
        if key == "tools":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].name)

### 自訂 PII 類型
你可以透過提供 detector 參數來建立自訂的 PII 類型。這能讓你偵測到內建類型之外、符合你自身需求的特殊資料格式。

建立自訂偵測器有三種方式：

>- 正規表示式（Regex）字串：用來做簡單的模式匹配
>- 自訂函式（Custom function）：適合更複雜、需要驗證邏輯的偵測方式

#### 方法一：使用正規表示式（Regex pattern string）

In [ ]:
import re

agent1 = create_agent(
    model=model_gpt_oss,
    middleware=[
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
        ),
    ],
)

content = "我在設定 OpenAI API 的時候遇到問題，我的金鑰是 sk-1234567890abcdef1234567890abcdef，可以幫我檢查為什麼不能用嗎？"

agent_input = {"messages": [{"role": "user", "content": content}]}

for update in agent1.stream(
    agent_input,
    stream_mode="updates",
):
    # 只保留 model 和 tool 兩個 key
    for key, value in update.items():
        if key == "model":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].tool_calls)
        if key == "tools":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].name)

#### 方法二 使用預先編譯的正規表示式（Compiled regex pattern）

In [ ]:
agent2 = create_agent(
    model=model_gpt_oss,
    middleware=[
        PIIMiddleware(
            "身分證號碼",
            detector=re.compile(r"[A-Z]\d{9}"),
            strategy="mask",
        ),
    ],
)

content = dedent("""
請幫我整理這段訊息：
聯絡人：王小明
身分證號碼：E123456789
地址：台北市信義區
""")

agent_input = {"messages": [{"role": "user", "content": content}]}

for update in agent2.stream(
    agent_input,
    stream_mode="updates",
):
    # 只保留 model 和 tool 兩個 key
    for key, value in update.items():
        if key == "model":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].tool_calls)
        if key == "tools":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].name)

#### 方法三：自訂偵測函式（Custom detector function）


自訂偵測函式的格式要求如下：

這個函式必須接收一個字串（content），並回傳所有偵測到的結果。

回傳的內容需為「字典（dictionary）構成的清單（list）」，且每個字典必須包含以下欄位：

- text：被偵測到的文字內容
- start：該內容在原字串中的起始位置
- end：該內容在原字串中的結束位置

這種方式適合需要較複雜邏輯的情況，例如多步驟驗證、跨欄位比對、或結合外部規則的偵測。

In [ ]:
def detect_ssn(content: str) -> list[dict[str, str | int]]:
    """Detect SSN with validation.

    example: 123-45-6789
    
    Returns a list of dictionaries with 'text', 'start', and 'end' keys.
    """
    import re
    matches = []
    pattern = r"\d{3}-\d{2}-\d{4}"
    for match in re.finditer(pattern, content):
        ssn = match.group(0)
        # Validate: first 3 digits shouldn't be 000, 666, or 900-999
        first_three = int(ssn[:3])
        if first_three not in [0, 666] and not (900 <= first_three <= 999):
            matches.append({
                "text": ssn,
                "start": match.start(),
                "end": match.end(),
            })
    return matches

agent3 = create_agent(
    model=model_gpt_oss,
    middleware=[
        PIIMiddleware(
            "ssn",
            detector=detect_ssn,
            strategy="hash",
        ),
    ],
)

content =dedent("""
我的資料是：
姓名：John
SSN：123-45-6789
請幫我建立資料檔
""")

agent_input = {"messages": [{"role": "user", "content": content}]}

for update in agent3.stream(
    agent_input,
    stream_mode="updates",
):
    # 只保留 model 和 tool 兩個 key
    for key, value in update.items():
        if key == "model":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].tool_calls)
        if key == "tools":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].name)

### 模型後援機制（Model Fallback）

當主要模型發生錯誤或無法使用時，系統可以自動切換到其他備用模型。  
這樣的「後援機制」能提升系統的穩定性與彈性。

模型後援特別適用於以下情境：

- **打造更可靠的代理系統**：即使主要模型暫時無法使用，服務仍可正常運作。  
- **節省成本**：在不需要高階能力時，可自動改用更便宜的模型。  
- **多供應商備援**：例如同時支援 OpenAI、Anthropic 等不同的模型供應商，降低單一供應商風險。

In [ ]:
from langchain.agents.middleware import ModelFallbackMiddleware

model_backup_1 = ChatOpenAI(model='gpt-4o-mini', name='agent_model_backup_1')
model_backup_2 = ChatOllama(model="kimi-k2.6:cloud", temperature=0,
                              base_url='https://ollama.com', name='agent_model_backup_1')

agent = create_agent(
    model=model_gpt_oss,
    middleware=[
        ModelFallbackMiddleware(
           model_backup_1, model_backup_2
        ),
    ],
)

## Middleware - Guardrail

### OpenAI 內容審核（OpenAI Moderator）

OpenAI Moderator 可用於識別文字和圖片中可能具有危害性的內容。  
透過 **moderations endpoint**，你可以檢查文字或圖片是否含有有害內容。  
若發現危害性內容，系統可以採取對應措施，例如過濾內容，或對產生不當內容的使用者帳號進行干預。  
此 **moderation endpoint** 是免費使用的。

### 可使用的模型

1. **omni-moderation-latest**  
   - 支援更多分類選項  
   - 支援多模態輸入（文字 + 圖片）  

2. **image_base64**  
   - 適用於對圖片進行基於 Base64 的審核

In [ ]:
import base64
import io
from PIL import Image


def image_to_base64(image_path):
    """
    Convert an image file to base64 encoded string.

    Args:
        image_path: Path to the image file

    Returns:
        Base64 encoded string representation of the image
    """
    with Image.open(image_path) as image:
        # Save the Image to a Buffer
        if image.mode in ("RGBA", "P"):
            image = image.convert("RGB")

        buffered = io.BytesIO()
        image.save(buffered, format="JPEG")

        # Encode the Image to Base64
        image_str = base64.b64encode(buffered.getvalue())

    return image_str.decode('utf-8')

In [ ]:
import os

from openai import OpenAI

client = OpenAI()

image_base64 = image_to_base64("tutorial/week_6/exp_image.png")

# response = client.moderations.create(
#     model="omni-moderation-latest",
#     input=[
#         {"type": "text", "text": "裡面的內容為何?"},
#         {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_base64}",
#             }
#         },
#     ],
# )

In [ ]:
response

In [ ]:
response.results[0].categories.model_dump()

In [ ]:
response.results[0].category_scores.model_dump()

輸出結果會包含多個分類欄位（JSON 格式），用來說明輸入內容中是否包含某些類型的內容，以及模型對這些內容存在程度的判斷信心。

| 輸出分類 | 說明 |
|----------|------|
| **flagged** | 若模型判定內容可能有害，則為 true，否則為 false。 |
| **categories** | 包含各分類違規標記的字典。對於每個分類，若模型判定該分類被違反則為 true，否則為 false。 |
| **category_scores** | 包含各分類的分數字典，表示模型對該輸入是否違反 OpenAI 政策的信心程度。數值介於 0 到 1 之間，數值越高代表信心越高。 |
| **category_applied_input_types** | 顯示每個分類中被標記的輸入類型。例如，若影像與文字同時在 *violence/graphic* 分類中被標記，則該欄位會是 `["image", "text"]`。此欄位僅適用於 Omni 模型。 |

---

### 內容分類（Content Classification）

下表說明 moderation API 可偵測的內容類型，以及各分類支援的模型與輸入形式。

| 分類 | 說明 | 模型 | 輸入類型 |
|------|------|------|----------|
| **harassment** | 對任何對象表達、煽動或促進騷擾性言論的內容。 | 全部 | 僅文字 |
| **harassment/threatening** | 包含暴力或嚴重傷害威脅的騷擾內容。 | 全部 | 僅文字 |
| **hate** | 基於種族、性別、族群、宗教、國籍、性傾向、身心障礙或階級等特徵表達或煽動仇恨的內容。針對非受保護群體（例如西洋棋玩家）的仇恨則歸類為騷擾。 | 全部 | 僅文字 |
| **hate/threatening** | 針對受保護群體的仇恨內容，並包含暴力或嚴重傷害威脅。 | 全部 | 僅文字 |
| **illicit** | 提供如何從事非法行為的建議或指示，例如「如何行竊」。 | 僅 Omni | 僅文字 |
| **illicit/violent** | 與 illicit 類似，但包含暴力或取得武器的相關內容。 | 僅 Omni | 僅文字 |
| **self-harm** | 宣揚、鼓勵或描述自我傷害行為（如自殺、割傷、飲食失調等）。 | 全部 | 文字與影像 |
| **self-harm/intent** | 表達說話者正在或打算進行自我傷害行為的內容。 | 全部 | 文字與影像 |
| **self-harm/instructions** | 鼓勵或提供如何進行自我傷害的指示內容。 | 全部 | 文字與影像 |
| **sexual** | 用於引起性興奮或推廣性服務的內容（不包含性教育與健康）。 | 全部 | 文字與影像 |
| **sexual/minors** | 涉及未滿 18 歲個體的性相關內容。 | 全部 | 僅文字 |
| **violence** | 描述死亡、暴力或身體傷害的內容。 | 全部 | 文字與影像 |
| **violence/graphic** | 以具體細節描寫死亡、暴力或身體傷害的內容。 | 全部 | 文字與影像 |


In [ ]:
response.results[0]

### 在代理防護措施（agent guardrails）之前執行

你可以在代理正式進入防護邏輯（guardrails）之前，先加入自訂的中介軟體流程。  
這讓你能在早期階段攔截、修改或分析請求，包含：

- 在進入正式流程前，先檢查與調整使用者的輸入內容  
- 套用額外的驗證或安全檢查  
- 根據需求修改提示（prompt）、加入標記或額外資訊  
- 在其他邏輯執行前，對請求進行紀錄或追蹤  

這能讓整個代理流程更加可控、可擴充，也更容易滿足自訂需求。

In [ ]:
from typing import Any

from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime


# 客製化AgentMiddleware
class ContentFilterMiddleware(AgentMiddleware):

    def __init__(self):
        super().__init__()
        self.moderator = client.moderations

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        # Get the first user message
        if not state["messages"]:
            return None

    @hook_config()
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        # Agent finished reasoning, inspect the final answer
        last_message = state["messages"][-1]
        print(f"[after_agent] Final response: {last_message.content[:200]}...")
        return None
        
        content = state['messages'][-1].content
        
        response = self.moderator.create(
            model="omni-moderation-latest",
            input=content
        )

        print(response)
        
        if response.results[0].flagged:

            reason = {key: value for key, value in response.results[0].categories.model_dump().items() if value}
            
            return {
                    "messages": [{
                        "role": "assistant",
                        "content": dedent(f"""\
                        I cannot process requests containing inappropriate content. 
                        Because of {reason}
                        Please rephrase your request.
                        """
                        )
                    }],
                    "jump_to": "end"
                }

        return None
        

model_kimi = ChatOllama(model='kimi-k2.6:cloud',
                   base_url='https://ollama.com',
                   name='image_caption', temperature=0)

agent = create_agent(
    model=model_kimi,
    middleware=[
        ContentFilterMiddleware(),
    ],
)

In [ ]:
from textwrap import dedent

agent_input = {
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "裡面的內容為何?"},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{image_base64}"
                    }
                }
            ]
        }
    ]
}

for update in agent.stream(
    agent_input,
    stream_mode="updates",
):
    # # 只保留 model 和 tool 兩個 key
    # for key, value in update.items():
    #     if key == "model":
    #         print(f"{key}: ", value["messages"][0].content, value['messages'][0].tool_calls)
    #     if key == "tools":
    #         print(f"{key}: ", value["messages"][0].content, value['messages'][0].name)
    print(update)

In [ ]:
image_base64 = image_to_base64("tutorial/week_5/StellarBladeTachy-Nikke.png")

agent_input = {
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "裡面的內容為何?"},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{image_base64}"
                    }
                }
            ]
        }
    ]
}

In [ ]:
for update in agent.stream(
    agent_input,
    stream_mode="updates",
):
    # 只保留 model 和 tool 兩個 key
    for key, value in update.items():
        if key == "model":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].tool_calls)
        if key == "tools":
            print(f"{key}: ", value["messages"][0].content, value['messages'][0].name)

# after_agent Hook 使用說明

## 生命週期位置

`after_agent` 在 Agent 完成推理之後觸發，流程如下：

```
User
 │
 ▼
before_agent()
 │
 ▼
LLM / Agent reasoning
 │
 ├── Tool
 ├── Tool
 ├── Tool
 │
 ▼
after_agent()       ← 在這裡
 │
 ▼
Return response
```

## 可取得的資訊

| 資訊 | 說明 |
|---|---|
| Agent 最後回答 | `state["messages"][-1].content` |
| Tool 執行結果 | `state["messages"]` 中 role 為 `tool` 的訊息 |
| State | 完整的對話狀態，包含所有訊息歷史 |

## 基本寫法

```python
@hook_config()
def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    last_message = state["messages"][-1]
    print(last_message.content)
    return None
```

## can_jump_to 跳轉選項

`@hook_config(can_jump_to=[...])` 宣告此 hook 可以跳轉到哪些 graph 節點。若不指定則只能回傳 `None`（讓流程繼續）。

| `jump_to` 值 | 跳轉目標 | 效果 |
|---|---|---|
| `"end"` | 結束節點 | 直接終止，回傳當前 messages |
| `"agent"` | LLM 調用節點 | 強制進入模型推理 |
| `"tools"` | 工具執行節點 | 跳過模型，直接執行 tool calls |

### before_agent 中的 jump_to 場景

```python
@hook_config(can_jump_to=["end", "tools"])
def before_agent(self, state, runtime):
    # 內容違規 → 直接結束
    if flagged:
        return {"messages": [...], "jump_to": "end"}

    # 偵測到特定 pattern → 直接觸發 tool，不經 LLM
    if match_pattern:
        return {"messages": [...], "jump_to": "tools"}

    return None  # 正常進入 LLM
```

### after_agent 中的 jump_to 場景

```python
@hook_config(can_jump_to=["end", "agent"])
def after_agent(self, state, runtime):
    # 輸出違規 → 直接結束
    if unsafe:
        return {"messages": [AIMessage(content="Sorry...")], "jump_to": "end"}

    # 輸出不如預期 → 強制模型再思考一輪
    if needs_rethink:
        return {"messages": [...], "jump_to": "agent"}

    return None  # 正常結束
```

## 常見用途

### 1. Output Moderation（輸出審查）

ChatGPT 官方建議的架構：

```
User → LLM → Moderation → Return
```

若回答包含 Hate、Violence、Personal data 等不安全內容，可攔截並替換：

```python
@hook_config(can_jump_to=["end"])
def after_agent(self, state, runtime):
    last_message = state["messages"][-1]

    if unsafe:
        return {
            "messages": [
                AIMessage(content="Sorry, I can't answer that.")
            ],
            "jump_to": "end"
        }
    return None
```

### 2. Logging（日誌記錄）

```python
@hook_config()
def after_agent(self, state, runtime):
    save_to_database(
        question=user_question,
        answer=agent_answer
    )
    return None
```

### 3. Metrics（指標收集）

可在 `after_agent` 中收集：

- **Latency**：Agent 整體執行時間
- **Tool count**：呼叫了多少次工具
- **Token count**：消耗的 token 數量
- **Cost**：費用估算

### 4. Post Processing（後處理）

統一在回答末尾附加免責聲明或標記：

```python
@hook_config()
def after_agent(self, state, runtime):
    last_message = state["messages"][-1]
    last_message.content += "\n\nGenerated by Internal AI"
    return None
```

## 與 before_agent 的對比

| Hook | 觸發時機 | 典型用途 |
|---|---|---|
| `before_agent` | Agent 開始前 | 輸入過濾、內容審查、攔截違規請求 |
| `after_agent` | Agent 完成後 | 輸出審查、日誌、指標、後處理 |

## 完整 Middleware 範例

```python
class ContentFilterMiddleware(AgentMiddleware):

    def __init__(self):
        super().__init__()
        self.moderator = client.moderations

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state, runtime):
        """輸入過濾：攔截不當請求"""
        ...

    @hook_config(can_jump_to=["end", "agent"])
    def after_agent(self, state, runtime):
        """輸出審查 + 日誌記錄"""
        last_message = state["messages"][-1]
        print(f"[after_agent] {last_message.content[:200]}...")
        return None
```
